# ToDo

# Подготовка

## Настройка графики

In [ ]:
# windows.options(height=5.4, width=7)
oldpar <- par()
par(mar = c(8, 4, 1, 2), "xpd" = FALSE)
options(repr.plot.height = 9, repr.plot.width = 12)
options(warn = -1)

## Библиотеки

In [ ]:
options(java.parameters = "-Xmx4096m")

require(readxl, quietly = TRUE, warn.conflicts = FALSE)

require(vcd, quietly = TRUE, warn.conflicts = FALSE)
require(coin, quietly = TRUE, warn.conflicts = FALSE)
# independence_test
require(agricolae, quietly = TRUE, warn.conflicts = FALSE)
# HSD.test
require(pgirmess, quietly = TRUE, warn.conflicts = FALSE)
# kruskalmc
require(nortest, quietly = TRUE, warn.conflicts = FALSE)
# for normality test in case of N>5000. ad.test -- Anderson-Darling normality test
require(RcmdrMisc, quietly = TRUE, warn.conflicts = FALSE)
# numSumm

require(beeswarm, quietly = TRUE, warn.conflicts = FALSE)
require(lattice, quietly = TRUE, warn.conflicts = FALSE)
require(mosaic, quietly = TRUE, warn.conflicts = FALSE)
require(ggplot2, quietly = TRUE, warn.conflicts = FALSE)
require(ggpubr, quietly = TRUE, warn.conflicts = FALSE)
# ggqqplot
# require(ggExtra, quietly = TRUE, warn.conflicts = FALSE);
# require(gridExtra, quietly = TRUE, warn.conflicts = FALSE);
# require(ggfortify, quietly = TRUE, warn.conflicts = FALSE);
require(ggalluvial, quietly = TRUE)
# flow diagramm
require(hrbrthemes, quietly = TRUE, warn.conflicts = FALSE)
# ggparcoord
require(GGally, quietly = TRUE, warn.conflicts = FALSE)
# ggparcoord
require(viridis, quietly = TRUE, warn.conflicts = FALSE)
# ggparcoord


require(rstatix, quietly = TRUE)
# identify_outliers
require(dplyr, quietly = TRUE, warn.conflicts = FALSE)
require(tidyr, quietly = TRUE, warn.conflicts = FALSE)
require(tidycmprsk, quietly = TRUE, warn.conflicts = FALSE)
# require(tidyverse, quietly = TRUE, warn.conflicts = FALSE);

require(IRdisplay, quietly = TRUE, warn.conflicts = FALSE)
require(repr, quietly = TRUE, warn.conflicts = FALSE)
# require(TeachingDemos, quietly = TRUE, warn.conflicts = FALSE);
# require(Rmisc, quietly = TRUE, warn.conflicts = FALSE);
# require(ranger, quietly = TRUE, warn.conflicts = FALSE);
# require(Epi, quietly = TRUE, warn.conflicts = FALSE);
# require(car, quietly = TRUE, warn.conflicts = FALSE);
# require(exactRankTests, quietly = TRUE, warn.conflicts = FALSE);
# require(abind, quietly = TRUE, warn.conflicts = FALSE);
# require(mstate, quietly = TRUE, warn.conflicts = FALSE);
# require(gnm, quietly = TRUE, warn.conflicts = FALSE);
# require(multcomp, quietly = TRUE, warn.conflicts = FALSE);
# require(scales, quietly = TRUE, warn.conflicts = FALSE);
# require(Rcmdr, quietly = TRUE, warn.conflicts = FALSE);
# require(tigerstats, quietly = TRUE, warn.conflicts = FALSE);
# require(fpp3, quietly = TRUE, warn.conflicts = FALSE);
# require(tsibble, quietly = TRUE, warn.conflicts = FALSE);
# require(lubridate, quietly = TRUE, warn.conflicts = FALSE);
# require(cowplot, quietly = TRUE, warn.conflicts = FALSE);
# require(smplot2, quietly = TRUE, warn.conflicts = FALSE);
# require(biostat3, quietly = TRUE, warn.conflicts = FALSE);
# require(data.table, quietly = TRUE, warn.conflicts = FALSE);

## Функции

In [ ]:
build_sankey <- function(data, group, parameter, group_colors) {
  for (groupID in parameter) {
    groupSize <- nlevels(data[[groupID]])
    group_colors1 <- group_colors[1:groupSize]
    for (i in 2:nlevels(data[[group]])) {
      tmp_tab <- table(data[data[[group]] == levels(data[[group]])[i - 1], groupID], data[data[[group]] == levels(data[[group]])[i], groupID])
      colnames(tmp_tab) <- paste(str_pad(levels(data[[group]])[i], 2, "left", "0"), colnames(tmp_tab), sep = "_")
      rownames(tmp_tab) <- paste(str_pad(levels(data[[group]])[i - 1], 2, "left", "0"), rownames(tmp_tab), sep = "_")
      if (i == 2) {
        out_tab <- as.data.table(tmp_tab)
        colnames(out_tab) <- c("source", "target", "value")
      } else {
        out_tab <- rbind(out_tab, as.data.table(tmp_tab), use.names = FALSE)
      }
    }
    out_tab <- cbind(as.data.frame(out_tab), group = rep(group_colors1, ceiling(dim(out_tab)[1] / length(group_colors1)))[1:dim(out_tab)[1]])

    links <- data.frame(out_tab[out_tab$value != 0, ])
    links <- links[order(links$source), ]

    nodes <- data.frame(name = unique(c(as.character(links$source), as.character(links$target))))
    unique_status <- sort(unique(substring(nodes$name, unlist(gregexpr(".[0-9]+$", gsub("\\+", ".", nodes$name))) + 1)))
    nodes$group <- group_colors1[match(gsub(".*\\.([0-9]+)$", "\\1", gsub("\\+", ".", nodes$name)), unique_status)]
    nodes <- nodes[order(nodes$name), ]
    rowTotals <- unique(rbind(aggregate(links$value, by = list(links$source), FUN = sum), aggregate(links$value, by = list(links$target), FUN = sum)))
    names(rowTotals) <- c("group", "x")
    nodes$rowtotal <- rowTotals$x
    nodes <- transform(nodes, rowpc = ave(rowtotal, substring(nodes$name, 1, 3), FUN = prop.table) * 100)

    links$IDsource <- match(links$source, nodes$name) - 1
    links$IDtarget <- match(links$target, nodes$name) - 1

    color <- sprintf(
      "d3.scaleOrdinal().domain([%s]).range([%s]);",
      paste0("'", paste(nodes$name, collapse = "", ""), "'"),
      paste0("'", paste(nodes$group, collapse = "", ""), "'")
    )

    #    links$group = substring(links$group, 2)
    #    nodes$group = substring(nodes$group, 2)

    sn <- sankeyNetwork(
      Links = links, Nodes = nodes,
      Source = "IDsource", Target = "IDtarget", Value = "value",
      NodeID = "name", LinkGroup = "source", NodeGroup = "name",
      units = "cases",
      sinksRight = FALSE, nodeWidth = 30, nodePadding = 30,
      colourScale = color,
      fontSize = 14, fontFamily = "Arial"
    )
    print(sn)
  }
}

metrics <- c("b750", "b730", "b735", "b755", "b760", "b134", "b152", "d330", "m_d330", "m_d4103", "m_d4104", "m_d4550", "e110", "e1101", "e410", "e455", "e580", "s110", "m_s110")
# group_colors = c("#f6e1f7ff", "#ecaad6FF", "#de6bb8ff", "#cc2e97ff", "#9d40a3FF", "#a7157FF", "#82003EFF", "#5e002dFF")
group_colors <- c("#0084ffff", "#44bec7ff", "#ffc300ff", "#fa3c4cff", "#d696bbff", "#a7157fff", "#82003eff", "#5e002dff", "#000000ff")
metrics <- c("d330")
# build_sankey(preml, "time", metrics, group_colors)


build_sankeyP <- function(data, group, parameter, group_colors) {
  for (groupID in parameter) {
    groupSize <- nlevels(data[[groupID]])
    group_colors1 <- group_colors[1:groupSize]
    for (i in 2:nlevels(data[[group]])) {
      tmp_tab <- table(data[data[[group]] == levels(data[[group]])[i - 1], groupID], data[data[[group]] == levels(data[[group]])[i], groupID])
      colnames(tmp_tab) <- paste(str_pad(levels(data[[group]])[i], 2, "left", "0"), colnames(tmp_tab), sep = "_")
      rownames(tmp_tab) <- paste(str_pad(levels(data[[group]])[i - 1], 2, "left", "0"), rownames(tmp_tab), sep = "_")
      if (i == 2) {
        out_tab <- as.data.table(tmp_tab)
        colnames(out_tab) <- c("source", "target", "value")
      } else {
        out_tab <- rbind(out_tab, as.data.table(tmp_tab), use.names = FALSE)
      }
    }
    out_tab <- cbind(as.data.frame(out_tab), group = rep(group_colors1, ceiling(dim(out_tab)[1] / length(group_colors1)))[1:dim(out_tab)[1]])

    links <- data.frame(out_tab[out_tab$value != 0, ])
    links <- links[order(links$source), ]

    nodes <- data.frame(name = unique(c(as.character(links$source), as.character(links$target))))
    unique_status <- sort(unique(substring(nodes$name, unlist(gregexpr(".[0-9]+$", gsub("\\+", ".", nodes$name))) + 1)))
    nodes$group <- group_colors1[match(gsub(".*\\.([0-9]+)$", "\\1", gsub("\\+", ".", nodes$name)), unique_status)]
    nodes <- nodes[order(nodes$name), ]
    rowTotals <- unique(rbind(aggregate(links$value, by = list(links$source), FUN = sum), aggregate(links$value, by = list(links$target), FUN = sum)))
    names(rowTotals) <- c("group", "x")
    nodes$rowtotal <- rowTotals$x
    nodes <- transform(nodes, rowpc = ave(rowtotal, substring(nodes$name, 1, 3), FUN = prop.table) * 100)

    links$IDsource <- match(links$source, nodes$name) - 1
    links$IDtarget <- match(links$target, nodes$name) - 1

    link <- list(
      source = links$IDsource,
      target = links$IDtarget,
      value = links$value,
      color = links$group
    )
    node <- list(
      label = sprintf("%i (%2.1f%%)", nodes$rowtotal, nodes$rowpc),
      color = nodes$group,
      pad = 10,
      line = list(
        color = "black",
        width = 1
      ),
      thickness = 30
    )
    domain <- list(
      x = c(0, 1),
      y = c(0, 1)
    )

    p <- plot_ly(
      link = link,
      node = node,
      domain = domain,
      type = "sankey",
      orientation = "h",
      valueformat = " .0f "
      # , valuesuffix = " случаи"
      , alpha = 1,
      height = 650,
      width = 850
    )

    p <- config(
      p,
      scrollZoom = TRUE,
      responsive = TRUE
      # , staticPlot = TRUE
      # , displayModeBar = TRUE
      , displaylogo = FALSE
    ) %>% layout(
      autosize = TRUE,
      title = groupID,
      font = list(size = 14),
      margin = list(
        l = 50,
        r = 50,
        b = 0,
        t = 100
        # , pad = 4
        # , automargin = TRUE
      ),
      xaxis = list(title = "Sepal Length (cm)"),
      legend = list(x = 1, y = 0.5)
    )
    print(p)
  }
}

# group_colors = c("#0084ff99", "#44bec799", "#ffc30099", "#fa3c4c99", "#d696bb99", "#a7157f99", "#82003e99", "#5e002d99")
# group_colors = c("#0084ff99", "#44bec799", "#ffc30099", "#fa3c4c99", "#d696bb99", "#a7157f99", "#82003e99", "#5e002d99")
# group_colors = c("rgba(0,255,255,0.4)", "#44bec799", "#ffc30099", "#fa3c4c99", "#d696bb99", "#a7157f99", "#82003e99", "#5e002d99")
# group_colors = c("#0084ffff", "#44bec7FF", "#ffc300ff", "#fa3c4cff", "#d696bbff", "#a7157ff", "#82003eff", "#5e002dff")
metrics <- c("b750", "b730", "b735", "b755", "b760", "b134", "b152", "d330", "m_d330", "m_d4103", "m_d4104", "m_d4550", "e110", "e1101", "e410", "e455", "e580", "s110", "m_s110")
group_colors <- c("#0084ffff", "#44bec7ff", "#ffc300ff", "#fa3c4cff", "#d696bbff", "#a7157fff", "#82003eff", "#5e002dff", "#000000ff")
metrics <- c("b750")


# build_sankeyP(lorl, "time", metrics, group_colors)


build_sankey_strataG <- function(data, group, strata, parameter, group_colors, save_multiple, save_file) {
  if (save_multiple) {
    pdf(sprintf("%s.pdf", strata), onefile = TRUE, paper = "a4r", width = 20, height = 14, encoding = "KOI8-R")
  }
  for (groupID in parameter) {
    in_data <- data[!is.na(data[[groupID]]), ]
    p <- ggplot(
      in_data,
      aes(
        x = in_data[[group]],
        stratum = in_data[[groupID]],
        alluvium = in_data[["uid"]],
        fill = in_data[[groupID]]
      )
      # , width = 1980
      # , height = 1024
    ) +
      facet_wrap(
        in_data[[strata]],
        scales = "free",
        strip.position = "top",
        drop = TRUE
      ) +
      geom_flow(
        # stat = "alluvium"
        # , lode.guidance = "frontback"
        ,
        width = 0.27,
        color = "darkgray",
        curve_type = "arctangent" # "linear", "cubic", "quintic", "sine", "arctangent", and "sigmoid". "xspline"
      ) +
      geom_stratum(alpha = .5) +
      geom_text(
        stat = "flow",
        aes(
          label = sprintf(" %i ", after_stat(n)),
          hjust = (after_stat(flow) == "to"),
          vjust = (after_stat(flow) == "to")
        ),
        size = 4
      ) +
      geom_text(
        stat = "stratum",
        aes(
          label = sprintf("%i\n(%3.1f%%)", after_stat(n), after_stat(prop * 100))
          # , hjust = as.integer(after_stat(stratum)) - 2
        ),
        hjust = 0,
        size = 5,
        nudge_x = 0.2
      ) +
      # expand_limits(x = 0, y = 0) +
      xlab(
        group
      ) +
      labs(fill = groupID, size = 5) +
      theme(
        legend.position = "bottom",
        panel.background = element_rect(fill = NA),
        panel.border = element_blank(),
        panel.grid.major = element_blank(),
        panel.grid.minor = element_blank(),
        strip.text = element_text(size = 14), ,
        axis.ticks.y = element_blank(),
        axis.text.y = element_blank(),
        axis.text.x = element_text(size = 12),
        axis.title.x = element_text(size = 14),
        axis.line.x = element_line(size = 1, colour = "black", linetype = 1),
        legend.text = element_text(size = 14),
        legend.title = element_blank(),
        panel.spacing.x = unit(0.5, "lines"),
        text = element_text(family = "Arial Narrow")
      ) +
      scale_fill_manual(values = group_colors) +
      ggtitle(groupID)
    if (save_file) {
      ggsave(sprintf("%s_%s.png", strata, groupID), plot = p, width = 36, height = 24, unit = "cm", dpi = "print")
    }
    print(p)
  }
  if (save_multiple) {
    dev.off()
  }
}


build_sankey_strata <- function(data, group, strata, parameter, group_colors) {
  strata_data <- list()
  for (i in 1:nlevels(data[[strata]])) {
    tmp_data <- subset(data, data[[strata]] == levels(data[[strata]])[i])
    strata_data[[i]] <- tmp_data
  }
  for (groupID in parameter) {
    for (i in seq_along(strata_data)) {
      print(paste(levels(data[[strata]])[i], ", ", groupID))
      build_sankey(strata_data[[i]], group, groupID, group_colors)
    }
  }
}

metrics <- c("m_s110")
# metrics = c("b750", "b730", "b735", "b755", "b760", "b134", "b152", "d330", "m_d330", "m_d4103", "m_d4104", "m_d4550", "e110", "e1101", "e410", "e455", "e580", "m_s110")
group_colors <- c("#0084ff", "#44bec7", "#ffc300", "#fa3c4c", "#d696bb", "#a7157f", "#82003e", "#5e002d", "#000000")

# build_sankey_strataG(preml, "time", "Абилитация", metrics, group_colors, FALSE, TRUE)

## Данные

### Загрузка

In [ ]:
# sessionInfo()
# options(encoding = "UTF-8")
lor <- read_excel("C:\\Analysis\\OTOLARING\\Nidelko\\lor.xlsx", sheet = "данные")
lor <- as.data.frame(lor)

### Преобразование

#### Исключение данных

In [ ]:
# Filter patients with age >= 70
# lor = lor %>%
#     filter(
#         возраст < 70
#         )

# SNOT 22
# lor = lor[, -305:-392]

# риноцитограмма.лейкоциты исходные
# lor = lor[, c(-276:-279, -236:-239, -200:-203, -156:-159)]
# даты
# lor = lor[, c(-34, -156:-159, -200:-203, -236:-239, -276:-279)]
# lor = lor[, -4]

#### Контрасты

In [ ]:
lor$группа <- factor(lor$группа, c("ОГ1", "ОГ2", "КГ"))
levels(lor$группа)[1] <- "ОГ"
levels(lor$группа)[2] <- "ОГ"

lor$пол <- factor(lor$пол)

lor$морфология <- factor(lor$морфология)

lor$"аллергия.домашняя пыль" <- factor(lor$"аллергия.домашняя пыль")
lor$"аллергия.растения" <- factor(lor$"аллергия.растения")
lor$"аллергия.пылевые клещи" <- factor(lor$"аллергия.пылевые клещи")
lor$"аллергия.животные" <- factor(lor$"аллергия.животные")
lor$"лекарственная непереносимость" <- factor(lor$"лекарственная непереносимость")

lor$"хронический ринит" <- factor(lor$"хронический ринит")
lor$"АР" <- factor(lor$"АР")
lor$"БА" <- factor(lor$"БА")
lor$"искривление носовой перегородки" <- factor(lor$"искривление носовой перегородки")

lor$"место взятия СК" <- factor(lor$"место взятия СК")
lor$"МСК" <- factor(lor$"МСК")
lor$"повторная аллотрансплантация" <- factor(lor$"повторная аллотрансплантация")
lor$"введение.верхние отделы перегородки" <- factor(lor$"введение.верхние отделы перегородки")
lor$"введение.средняя носовая раковина" <- factor(lor$"введение.средняя носовая раковина")
lor$"введение.латеральная стенка полости носа" <- factor(lor$"введение.латеральная стенка полости носа")
lor$"введение.верхий отдел нижних носовых раковин" <- factor(lor$"введение.верхий отдел нижних носовых раковин")
lor$"аллергопроба" <- factor(lor$"аллергопроба")
lor$"срок оценки результата" <- factor(lor$"срок оценки результата", c("1-3 сутки", "6-7 сутки"))
lor$"постинъекционный отек" <- factor(lor$"постинъекционный отек")
lor$"постиънекционный отек.2" <- factor(lor$"постиънекционный отек.2")
lor$"цвет.сутки" <- factor(lor$"цвет.сутки", c("бледно-розовый", "другой"))
lor$"цвет.2" <- factor(lor$"цвет.2", c("бледно-розовый", "другой"))

lor$"верхнечелюстная" <- factor(lor$"верхнечелюстная", c("нет", "односторонняя", "двустороняя"))
lor$"этмоидальная" <- factor(lor$"этмоидальная", c("нет", "односторонняя", "двустороняя"))
lor$"фронтальная" <- factor(lor$"фронтальная", c("нет", "односторонняя", "двустороняя"))
lor$"сфеноидальная" <- factor(lor$"сфеноидальная", c("нет", "односторонняя", "двустороняя"))
lor$"вазотомия нижних носовых раковин" <- factor(lor$"вазотомия нижних носовых раковин")
lor$"операция на перегородке носа" <- factor(lor$"операция на перегородке носа")
lor$FESS <- factor(lor$FESS)

lor$"топические ГКС.0" <- factor(lor$"топические ГКС.0")
lor$"топические ГКС.1" <- factor(lor$"топические ГКС.1")
lor$"топические ГКС.2" <- factor(lor$"топические ГКС.2")
lor$"топические ГКС.3" <- factor(lor$"топические ГКС.3")

lor$"затруднение носового дыхания.0" <- factor(lor$"затруднение носового дыхания.0", c("да", "нет", "не выполнялось"))
lor$"окрашенные выделения из носа.0" <- factor(lor$"окрашенные выделения из носа.0", c("да", "нет", "не выполнялось"))
lor$"прозрачные выделения из носа.0" <- factor(lor$"прозрачные выделения из носа.0", c("да", "нет", "не выполнялось"))
lor$"головная боль в проекции онп.0" <- factor(lor$"головная боль в проекции онп.0", c("да", "нет", "не выполнялось"))
lor$"гипосмия.0" <- factor(lor$"гипосмия.0", c("да", "нет", "не выполнялось"))
lor$"аносмия.0" <- factor(lor$"аносмия.0", c("да", "нет", "не выполнялось"))
lor$"затруднение носового дыхания.1" <- factor(lor$"затруднение носового дыхания.1")
lor$"окрашенные выделения из носа.1" <- factor(lor$"окрашенные выделения из носа.1")
lor$"прозрачные выделения из носа.1" <- factor(lor$"прозрачные выделения из носа.1")
lor$"головная боль в проекции онп.1" <- factor(lor$"головная боль в проекции онп.1")
lor$"гипосмия.1" <- factor(lor$"гипосмия.1")
lor$"аносмия.1" <- factor(lor$"аносмия.1")
lor$"затруднение носового дыхания.2" <- factor(lor$"затруднение носового дыхания.2")
lor$"окрашенные выделения из носа.2" <- factor(lor$"окрашенные выделения из носа.2")
lor$"прозрачные выделения из носа.2" <- factor(lor$"прозрачные выделения из носа.2")
lor$"головная боль в проекции онп.2" <- factor(lor$"головная боль в проекции онп.2")
lor$"гипосмия.2" <- factor(lor$"гипосмия.2")
lor$"аносмия.2" <- factor(lor$"аносмия.2")
lor$"затруднение носового дыхания.3" <- factor(lor$"затруднение носового дыхания.3")
lor$"окрашенные выделения из носа.3" <- factor(lor$"окрашенные выделения из носа.3")
lor$"прозрачные выделения из носа.3" <- factor(lor$"прозрачные выделения из носа.3")
lor$"головная боль в проекции онп.3" <- factor(lor$"головная боль в проекции онп.3")
lor$"гипосмия.3" <- factor(lor$"гипосмия.3")
lor$"аносмия.3" <- factor(lor$"аносмия.3")
lor$"АР.чихание.0" <- factor(lor$"АР.чихание.0")
lor$"АР.заложенность носа.0" <- factor(lor$"АР.заложенность носа.0")
lor$"АР.зуд в полости носа.0" <- factor(lor$"АР.зуд в полости носа.0")
lor$"АР.обильное водянистое отделяемое.0" <- factor(lor$"АР.обильное водянистое отделяемое.0")
lor$"АР итог.0" <- factor(lor$"АР итог.0")
lor$"АР.чихание.6" <- factor(lor$"АР.чихание.6")
lor$"АР.заложенность носа.6" <- factor(lor$"АР.заложенность носа.6")
lor$"АР.зуд в полости носа.6" <- factor(lor$"АР.зуд в полости носа.6")
lor$"АР.обильное водянистое отделяемое.6" <- factor(lor$"АР.обильное водянистое отделяемое.6")
lor$"АР итог.6" <- factor(lor$"АР итог.6")
lor$"АР.чихание.12" <- factor(lor$"АР.чихание.12")
lor$"АР.заложенность носа.12" <- factor(lor$"АР.заложенность носа.12")
lor$"АР.зуд в полости носа.12" <- factor(lor$"АР.зуд в полости носа.12")
lor$"АР.обильное водянистое отделяемое.12" <- factor(lor$"АР.обильное водянистое отделяемое.12")
lor$"АР итог.12" <- factor(lor$"АР итог.12")
lor$"АР.антигистаминный.0" <- factor(lor$"АР.антигистаминный.0")
lor$"АР.ИГКС.0" <- factor(lor$"АР.ИГКС.0")
lor$"АР лекарства итог.0" <- factor(lor$"АР лекарства итог.0")
lor$"АР.антигистаминный.6" <- factor(lor$"АР.антигистаминный.6")
lor$"АР.ИГКС.6" <- factor(lor$"АР.ИГКС.6")
lor$"АР лекарства итог.6" <- factor(lor$"АР лекарства итог.6")
lor$"АР.антигистаминный.12" <- factor(lor$"АР.антигистаминный.12")
lor$"АР.ИГКС.12" <- factor(lor$"АР.ИГКС.12")
lor$"АР лекарства итог.12" <- factor(lor$"АР лекарства итог.12")
lor$"прозраное отделяемое.0" <- factor(lor$"прозраное отделяемое.0")
lor$"прозраное отделяемое.1" <- factor(lor$"прозраное отделяемое.1")
lor$"прозраное отделяемое.2" <- factor(lor$"прозраное отделяемое.2")
lor$"прозраное отделяемое.3" <- factor(lor$"прозраное отделяемое.3")
lor$"окрашенное отделяемое в носовых ходах.0" <- factor(lor$"окрашенное отделяемое в носовых ходах.0")
lor$"окрашенное отделяемое в носовых ходах.1" <- factor(lor$"окрашенное отделяемое в носовых ходах.1")
lor$"окрашенное отделяемое в носовых ходах.2" <- factor(lor$"окрашенное отделяемое в носовых ходах.2")
lor$"окрашенное отделяемое в носовых ходах.3" <- factor(lor$"окрашенное отделяемое в носовых ходах.3")
lor$"проходимость соустий ВЧП.0" <- factor(lor$"проходимость соустий ВЧП.0")
lor$"проходимость соустий ВЧП.1" <- factor(lor$"проходимость соустий ВЧП.1")
lor$"проходимость соустий ВЧП.2" <- factor(lor$"проходимость соустий ВЧП.2")
lor$"проходимость соустий ВЧП.3" <- factor(lor$"проходимость соустий ВЧП.3")

lor$"распространенность полипозного процесса КТ" <- factor(lor$"распространенность полипозного процесса КТ")
lor$"лобные пазухи" <- factor(lor$"лобные пазухи")
lor$"клиновидные пазухи" <- factor(lor$"клиновидные пазухи")
lor$"решетчатый лабиринт передний" <- factor(lor$"решетчатый лабиринт передний")
lor$"решетчатый лабиринт задний" <- factor(lor$"решетчатый лабиринт задний")
lor$"верхнечелюстные пазухи" <- factor(lor$"верхнечелюстные пазухи")
lor$"остиомеатальный комплекс" <- factor(lor$"остиомеатальный комплекс")

lor$"отделяемое" <- factor(lor$"отделяемое")
lor$"корки" <- factor(lor$"корки")
lor$"отек" <- factor(lor$"отек")

In [ ]:
# replace NA with "-"
lor$"грибы.0" <- lor$"грибы.0" %>%
  replace_na("-")
lor$"грибы.1" <- lor$"грибы.1" %>%
  replace_na("-")
lor$"грибы.2" <- lor$"грибы.2" %>%
  replace_na("-")
lor$"грибы.3" <- lor$"грибы.3" %>%
  replace_na("-")

# replace "++" with "+"
lor$"грибы.0" <- ifelse(lor$"грибы.0" == "++", "+", lor$"грибы.0")
lor$"грибы.1" <- ifelse(lor$"грибы.1" == "++", "+", lor$"грибы.1")
lor$"грибы.2" <- ifelse(lor$"грибы.2" == "++", "+", lor$"грибы.2")
lor$"грибы.3" <- ifelse(lor$"грибы.3" == "++", "+", lor$"грибы.3")

lor$"грибы.0" <- factor(lor$"грибы.0", c("-", "+"))
lor$"грибы.1" <- factor(lor$"грибы.1", c("-", "+"))
lor$"грибы.2" <- factor(lor$"грибы.2", c("-", "+"))
lor$"грибы.3" <- factor(lor$"грибы.3", c("-", "+"))

In [ ]:
# "нет", "кб" -- "кб+"
lor$"флора.0" <- factor(lor$"флора.0", c("нет", "кб", "кб+", "кб++", "кб+++"))
lor$"флора.1" <- factor(lor$"флора.1", c("нет", "кб", "кб+", "кб++", "кб+++"))
lor$"флора.2" <- factor(lor$"флора.2", c("нет", "кб", "кб+", "кб++", "кб+++"))
lor$"флора.3" <- factor(lor$"флора.3", c("нет", "кб", "кб+", "кб++", "кб+++"))
levels(lor$"флора.0")[2] <- "кб+"
levels(lor$"флора.0")[1] <- "кб+"
levels(lor$"флора.1")[2] <- "кб+"
levels(lor$"флора.1")[1] <- "кб+"
levels(lor$"флора.2")[2] <- "кб+"
levels(lor$"флора.2")[1] <- "кб+"
levels(lor$"флора.3")[2] <- "кб+"
levels(lor$"флора.3")[1] <- "кб+"

In [ ]:
lor <- lor %>%
  separate("эпителиальные клетки0.0", into = c("low", "high"), sep = "-") %>%
  mutate(across(.cols = c(low, high), .fns = as.numeric)) %>%
  rowwise() %>%
  mutate("эпителиальные клетки.0" = mean(c(low, high)))
lor <- lor %>%
  separate("эпителиальные клетки0.1", into = c("low", "high"), sep = "-") %>%
  mutate(across(.cols = c(low, high), .fns = as.numeric)) %>%
  rowwise() %>%
  mutate("эпителиальные клетки.1" = mean(c(low, high)))
lor <- lor %>%
  separate("эпителиальные клетки0.2", into = c("low", "high"), sep = "-") %>%
  mutate(across(.cols = c(low, high), .fns = as.numeric)) %>%
  rowwise() %>%
  mutate("эпителиальные клетки.2" = mean(c(low, high)))
lor <- lor %>%
  separate("эпителиальные клетки0.3", into = c("low", "high"), sep = "-") %>%
  mutate(across(.cols = c(low, high), .fns = as.numeric)) %>%
  rowwise() %>%
  mutate("эпителиальные клетки.3" = mean(c(low, high)))
lor <- lor %>%
  select(-low, -high)

In [ ]:
lor <- lor %>%
  separate("риноцитограмма.лейкоциты0.0", into = c("low", "high"), sep = "-") %>%
  mutate(across(.cols = c(low, high), .fns = as.numeric)) %>%
  rowwise() %>%
  mutate("риноцитограмма.лейкоциты.0" = mean(c(low, high)))
lor <- lor %>%
  separate("риноцитограмма.лейкоциты0.1", into = c("low", "high"), sep = "-") %>%
  mutate(across(.cols = c(low, high), .fns = as.numeric)) %>%
  rowwise() %>%
  mutate("риноцитограмма.лейкоциты.1" = mean(c(low, high)))
lor <- lor %>%
  separate("риноцитограмма.лейкоциты0.2", into = c("low", "high"), sep = "-") %>%
  mutate(across(.cols = c(low, high), .fns = as.numeric)) %>%
  rowwise() %>%
  mutate("риноцитограмма.лейкоциты.2" = mean(c(low, high)))
lor <- lor %>%
  separate("риноцитограмма.лейкоциты0.3", into = c("low", "high"), sep = "-") %>%
  mutate(across(.cols = c(low, high), .fns = as.numeric)) %>%
  rowwise() %>%
  mutate("риноцитограмма.лейкоциты.3" = mean(c(low, high)))
lor <- lor %>%
  select(-low, -high)

In [ ]:
# Шкала Лунд-Кеннеди
lor$"полипы.0" <- factor(lor$"полипы.0", c("0", "1", "2", "3", "4"))
lor$"полипы.1" <- factor(lor$"полипы.1", c("0", "1", "2", "3", "4"))
lor$"полипы.2" <- factor(lor$"полипы.2", c("0", "1", "2", "3", "4"))
lor$"полипы.3" <- factor(lor$"полипы.3", c("0", "1", "2", "3", "4"))
lor$"отек слизистой.0" <- factor(lor$"отек слизистой.0", c("0", "1", "2", "3", "4"))
lor$"отек слизистой.1" <- factor(lor$"отек слизистой.1", c("0", "1", "2", "3", "4"))
lor$"отек слизистой.2" <- factor(lor$"отек слизистой.2", c("0", "1", "2", "3", "4"))
lor$"отек слизистой.3" <- factor(lor$"отек слизистой.3", c("0", "1", "2", "3", "4"))
lor$"выделения из носа.0" <- factor(lor$"выделения из носа.0", c("0", "1", "2", "3", "4"))
lor$"выделения из носа.1" <- factor(lor$"выделения из носа.1", c("0", "1", "2", "3", "4"))
lor$"выделения из носа.2" <- factor(lor$"выделения из носа.2", c("0", "1", "2", "3", "4"))
lor$"выделения из носа.3" <- factor(lor$"выделения из носа.3", c("0", "1", "2", "3", "4"))

lor$"шкала итог0.0" <- lor$"шкала итог.0"
lor$"шкала итог0.1" <- lor$"шкала итог.1"
lor$"шкала итог0.2" <- lor$"шкала итог.2"
lor$"шкала итог0.3" <- lor$"шкала итог.3"

lor$"шкала итог.0" <- cut(lor$"шкала итог.0", c(0, 3, 6, 8, 12), labels = c("0-3", "4-6", "7-8", "9-12"), include.lowest = TRUE, ordered_result = TRUE)
lor$"шкала итог.1" <- cut(lor$"шкала итог.1", c(0, 3, 6, 8, 12), labels = c("0-3", "4-6", "7-8", "9-12"), include.lowest = TRUE, ordered_result = TRUE)
lor$"шкала итог.2" <- cut(lor$"шкала итог.2", c(0, 3, 6, 8, 12), labels = c("0-3", "4-6", "7-8", "9-12"), include.lowest = TRUE, ordered_result = TRUE)
lor$"шкала итог.3" <- cut(lor$"шкала итог.3", c(0, 3, 6, 8, 12), labels = c("0-3", "4-6", "7-8", "9-12"), include.lowest = TRUE, ordered_result = TRUE)

#### Нормы

In [ ]:
oak.norm <- data.frame(
  name = c(
    "ОАК.эритроциты", "ОАК.гемоглобин", "ОАК.тромбоциты", "ОАК.лейкоциты", "ОАК.палочкоядерные", "ОАК.сегментоядерные", "ОАК.лимфоциты",
    "ОАК.моноциты", "ОАК.эозинофилы", "ОАК.СОЭ"
  ),
  min = c(3.8, 115, 150, 3.5, 1, 47, 18, 4, 0.5, 2),
  max = c(5.2, 152, 400, 10, 6, 72, 45, 12, 7, 15)
)

#### Длинная таблица

In [ ]:
lor <- as.data.frame(lor)
# print(names(lor))
lorl <- reshape(lor,
  direction = "long",
  idvar = "uid"
  # , drop = c(  "id", "рождение", "возраст", "длительность заболевания", "удаление полипов в анамнезе", "аллергия.домашняя пыль", "аллергия.растения"
  #            , "аллергия.пылевые клещи", "аллергия.животные", "лекарственная непереносимость", "хронический ринит", "АР", "БА"
  #            , "искривление носовой перегородки", "наблюдение.0", "дата операции", "наблюдение.1", "дата введения СК", "наблюдение.2", "наблюдение.3"
  #            , "место взятия СК", "МСК", "объем суспензии СК", "количествено клеток", "повторная аллотрансплантация"
  #            , "введение.верхние отделы перегородки", "введение.средняя носовая раковина", "введение.латеральная стенка полости носа"
  #            , "введение.верхий отдел нижних носовых раковин", "аллергопроба", "дата оценки результата", "срок оценки результата"
  #            , "постинъекционный отек", "постиънекционный отек 1-2мес", "цвет.2", "цвет.3", "оперированные пазухи", "верхнечелюстная", "этмоидальная"
  #            , "фронтальная", "сфеноидальная", "FESS", "вазотомия нижних носовых раковин", "операция на перегородке носа"
  #            , "системные ГКС.0", "системные ГКС.1", "системные ГКС.2", "системные ГКС.3"
  #            , "топические ГКС.0", "топические ГКС.1", "топические ГКС.2", "топические ГКС.3"
  #            , "суточная доза.0", "суточная доза.1", "суточная доза.2", "суточная доза.3"
  #            , "кратность приема.0", "кратность приема.1", "кратность приема.2", "кратность приема.3"
  #            , "кратность курсов.0", "кратность курсов.1", "кратность курсов.2", "кратность курсов.3"
  #            , "длительность курса.0", "длительность курса.1", "длительность курса.2", "длительность курса.3"
  #            , "АР.чихание.0", "АР.заложенность носа.0", "АР.зуд в полости носа.0", "АР.обильное водянистое отделяемое.0", "АР итог.0"
  #            , "АР.чихание.6", "АР.заложенность носа.6", "АР.зуд в полости носа.6", "АР.обильное водянистое отделяемое.6", "АР итог.6"
  #            , "АР.чихание.12", "АР.заложенность носа.12", "АР.зуд в полости носа.12", "АР.обильное водянистое отделяемое.12", "АР итог.12"
  #            , "АР.антигистаминный.0", "АР.ИГКС.0", "АР лекарства итог.0", "АР.антигистаминный.6", "АР.ИГКС.6", "АР лекарства итог.6"
  #            , "АР.антигистаминный.12", "АР.ИГКС.12", "АР лекарства итог.12"
  #            , "дата ОАК 1", "дата ОАК 2", "дата ОАК 3", "дата ОАК 4"
  #            , "дата БХАК.0", "дата БХАК.1", "дата БХАК.2", "дата БХАК.3", "дата.0", "дата.1", "дата.2", "дата.3"
  #            , "дата риноцитограммы.0", "дата риноцитограммы.1", "дата риноцитограммы.2", "дата риноцитограммы.3"
  #           )
  , sep = ".",
  varying = list(
    48:51, 52:55, 56:59, 60:63, 64:67, 68:71,
    72:75, 76:79, 80:83, 84:87, 88:91, 92:95,
    120:123, 124:127, 128:131,
    132:135, 136:139, 140:143, 144:147, 402:405,
    160:163, 164:167, 168:171, 172:175, 176:179, 180:183,
    184:187, 188:191, 192:195, 196:199,
    204:207, 398:401, 208:211, 212:215, 216:219, 220:223,
    224:227,
    232:235, 236:239, 240:243, 244:247, 248:251, 252:255,
    256:259, 260:263, 264:267,
    272:275, 276:279, 280:283, 284:287, 288:291, 292:295,
    296:299, 300:303
  ),
  v.names = c(
    "системные ГКС", "топические ГКС", "суточная доза", "кратность приема", "кратность курсов", "длительность курса",
    "затруднение носового дыхания", "окрашенные выделения из носа", "прозрачные выделения из носа", "головная боль в проекции онп", "гипосмия", "аносмия",
    "прозраное отделяемое", "окрашенное отделяемое в носовых ходах", "проходимость соустий ВЧП",
    "полипы", "отек слизистой", "выделения из носа", "шкала итог", "шкала итог0",
    "ОАК.эритроциты", "ОАК.гемоглобин", "ОАК.тромбоциты", "ОАК.лейкоциты", "ОАК.палочкоядерные", "ОАК.сегментоядерные",
    "ОАК.лимфоциты", "ОАК.моноциты", "ОАК.эозинофилы", "ОАК.СОЭ",
    "грибы", "эпителиальные клетки", "флора", "риноцитограмма.лейкоциты", "риноцитограмма.нейтрофилы", "риноцитограмма.лимфоциты",
    "риноцитограмма.эозинофилы",
    "белок", "мочевина", "креатинин", "АСАТ", "АЛАТ", "билирубин",
    "глюкоза", "СРБ", "РФ",
    "левая половина вдох скорость", "левая половина выдох, скорость", "правая половина вдох, скорость", "правая половина выдох, скорость", "сопротивление 150 ПА правая вдох", "сопротивление 150 ПА правая выдох",
    "сопротивление 150 ПА левая вдох", "сопротивление 150 ПА левая выдох"
  ),
  timevar = "time",
  times = c("до операции", "до СК", "3 мес.", "1 год")
)
lorl$time <- factor(lorl$time, c("до операции", "до СК", "3 мес.", "1 год"))

# Filter data for "до СК"
lorl <- filter(lorl, time != "до СК")
levels(lorl$time)[2] <- NA

# print(names(lorl))

### Отбор данных

In [ ]:
lor <- lor %>%
  filter(морфология == "да")
lor <- as.data.frame(lor)
attach(lor)

### Подключение

In [ ]:
try(detach(lor), silent = TRUE)
attach(lor)

### Интервалы между обследованиями

In [ ]:
`интервал.0` <- difftime(`дата операции`, `наблюдение.0`, units = "days")
`интервал.2` <- difftime(`дата введения СК`, `дата операции`, units = "days")
`интервал.3` <- difftime(`наблюдение.2`, `дата введения СК`, units = "days")
`интервал.4` <- difftime(`наблюдение.3`, `наблюдение.2`, units = "days")
`наблюдение.0-наблюдение.3` <- difftime(`наблюдение.3`, `наблюдение.0`, units = "days")
`операция-наблюдение.2` <- difftime(`наблюдение.2`, `дата операции`, units = "days")
`операция-наблюдение.3` <- difftime(`наблюдение.3`, `дата операции`, units = "days")
`операция-наблюдение.2` <- trunc(`операция-наблюдение.2` / 30)
`операция-наблюдение.3` <- trunc(`операция-наблюдение.3` / 30)

In [ ]:
cat("\nИнтервал между обращением и операцией")
numSummary(`интервал.0`,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, 0.1, .25, .5, .75, 0.9, 1), groups = группа
)
favstats(`интервал.0` ~ группа)
try(histogram(~ as.numeric(`интервал.0`) | группа, ylab = "% от общего количества", col = "light blue", xlab = "интервал, дни"))

In [ ]:
cat("\nИнтервал между операцией и введением СК")
numSummary(`интервал.2`,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, 0.1, .25, .5, .75, 0.9, 1), groups = группа
)
favstats(`интервал.2` ~ группа)
try(histogram(~ as.numeric(`интервал.2`), ylab = "% от общего количества", col = "light blue", xlab = "интервал, дни"))

In [ ]:
cat("\nИнтервал между введением СК и промежуточным наблюдением")
numSummary(`интервал.3`,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, 0.1, .25, .5, .75, 0.9, 1), groups = группа
)
favstats(`интервал.3` ~ группа)
try(histogram(~ as.numeric(`интервал.3`) | группа, ylab = "% от общего количества", col = "light blue", xlab = "интервал, дни"))

In [ ]:
cat("\nИнтервал между промежуточным наблюдением и последним наблюдением")
numSummary(`интервал.4`,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, 0.1, .25, .5, .75, 0.9, 1), groups = группа
)
favstats(`интервал.4` ~ группа)
try(histogram(~ as.numeric(`интервал.4`) | группа, ylab = "% от общего количества", col = "light blue", xlab = "интервал, дни"))

In [ ]:
cat("\nИнтервал между операцией и промежуточным наблюдением")
# numSummary(`операция-наблюдение.2`, statistics = c("mean", "sd", "IQR", "quantiles"),
#            quantiles = c(0, 0.1, .25, .5, .75, 0.9, 1))
favstats(`операция-наблюдение.2`)
numSummary(as.numeric(`операция-наблюдение.2`),
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, 0.1, .25, .5, .75, 0.9, 1)
)
try(histogram(~ as.numeric(`операция-наблюдение.2`), ylab = "% от общего количества", col = "light blue", xlab = "интервал, мес."))

In [ ]:
cat("\nИнтервал между операцией и промежуточным наблюдением")
# numSummary(`операция-наблюдение.2`, statistics = c("mean", "sd", "IQR", "quantiles"),
#            quantiles = c(0, 0.1, .25, .5, .75, 0.9, 1))
favstats(`операция-наблюдение.2`)

numSummary(`операция-наблюдение.2`,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, 0.1, .25, .5, .75, 0.9, 1), groups = группа
)
favstats(`операция-наблюдение.2` ~ группа)
try(histogram(~ as.numeric(`операция-наблюдение.2`) | группа, ylab = "% от общего количества", col = "light blue", xlab = "интервал, мес."))

In [ ]:
cat("\nИнтервал между операцией и последним наблюдением")
favstats(`операция-наблюдение.3`)

numSummary(as.numeric(`операция-наблюдение.3`),
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, 0.1, .25, .5, .75, 0.9, 1)
)
favstats(`операция-наблюдение.3` ~ группа)
try(histogram(~ as.numeric(`операция-наблюдение.3`), ylab = "% от общего количества", col = "light blue", xlab = "интервал, мес."))

In [ ]:
cat("\nИнтервал между операцией и последним наблюдением")
favstats(`операция-наблюдение.3`)

numSummary(`операция-наблюдение.3`,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, 0.1, .25, .5, .75, 0.9, 1), groups = группа
)
favstats(`операция-наблюдение.3` ~ группа)
try(histogram(~ as.numeric(`операция-наблюдение.3`) | группа, ylab = "% от общего количества", col = "light blue", xlab = "интервал, мес."))

# Общий анализ

## возраст

### Все

In [ ]:
name <- "возраст"
values <- lor[[name]]

In [ ]:
histogram(~values,
  ylab = "% от общего количества", col = "light blue", xlab = name,
  breaks = seq(20, 75, 5)
)

In [ ]:
numSummary(values, statistics = c("mean", "sd", "IQR", "quantiles"), quantiles = c(0, .25, .5, .75, 1))
favstats(values)

In [ ]:
qqmath(~values, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = name)

#### ad.test(values)

### пол

#### Исходные

##### Общее

In [ ]:
parname <- "пол"
parameter <- lor[[parname]]

In [ ]:
histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = name, breaks = seq(20, 75, 5))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = name)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(1.5, 20, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = F, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

##### Тест на нормальность

In [ ]:
for (i in 1:nlevels(lor[[parname]])) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[parname]] == levels(parameter)[i])
  print(shapiro.test(ss[[name]]))
}

In [ ]:
bartlett.test(lor[[name]] ~ parameter)

##### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))
}

##### Сравнение, распределение не нормальное

In [ ]:
wilcox.test(values ~ parameter)

In [ ]:
independence_test(values ~ parameter,
  data = lor,
  alternative = "greater",
  ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
  xtrafo = function(data) trafo(data, ordered_trafo = ff)
)

In [ ]:
independence_test(values ~ parameter,
  data = lor,
  alternative = "greater",
  ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
  xtrafo = function(data) trafo(data, ordered_trafo = ff)
)

### группа

#### Исходные

##### Общее

In [ ]:
parname <- "группа"
parameter <- lor[[parname]]

In [ ]:
histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = name)

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = name)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, 6, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

##### Тест на нормальность

In [ ]:
for (i in 1:nlevels(lor[[parname]])) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[parname]] == levels(parameter)[i])
  print(shapiro.test(ss[[name]]))
}

In [ ]:
bartlett.test(lor[[name]] ~ parameter)

##### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(lor[[parname]]) - 1)) {
    for (j in (i + 1):nlevels(lor[[parname]])) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(lor[[parname]])[i], `name`], lor[parameter == levels(lor[[parname]])[j], `name`]))
    }
  }
}

##### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(lor[[name]] ~ lor[[parname]]))
  print(kruskalmc(lor[[name]] ~ lor[[parname]]))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(lor[[parname]]) - 1)) {
    for (j in (i + 1):nlevels(lor[[parname]])) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[parname]])[i] | parameter == levels(lor[[parname]])[j])
      print(wilcox.test(ss[[name]] ~ ss[[parname]], exact = TRUE))
      print(independence_test(ss[[name]] ~ ss[[parname]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

### группа, пол

#### Исходные

##### Общее

In [ ]:
parname1 <- "группа"
parname2 <- "пол"
parameter1 <- lor[[parname1]]
parameter2 <- lor[[parname2]]

In [ ]:
histogram(~ values | parameter2 + parameter1, ylab = "% от общего количества", col = "light blue", xlab = name, breaks = seq(20, 75, 5))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = paste(parameter1, parameter2)
)
favstats(values ~ parameter2)

In [ ]:
qqmath(~ values | parameter2 + parameter1, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = name)

In [ ]:
beeswarm(values ~ parameter1 + parameter2, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = paste0(parname1, ", ", parname2))
boxplot(values ~ parameter1 + parameter2, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, 6, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

##### Тест на нормальность

In [ ]:
for (i in 1:nlevels(lor[[parname1]])) {
  cat("Группа —", levels(parameter1)[i])
  ss <- subset(lor, lor[[parname1]] == levels(parameter1)[i])
  print(shapiro.test(ss[[name]]))
}

In [ ]:
bartlett.test(lor[[name]] ~ parameter1)

##### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter1) < 3) {
  t.test(values ~ parameter1)
} else {
  agg1.aov <- aov(values ~ parameter1)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(lor[[parname]]) - 1)) {
    for (j in (i + 1):nlevels(lor[[parname]])) {
      cat("\nГруппы — ", levels(parameter1)[i], ", ", levels(parameter1)[j])
      print(t.test(lor[parameter1 == levels(lor[[parname]])[i], `name`], lor[parameter1 == levels(lor[[parname]])[j], `name`]))
    }
  }
}

##### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter1) < 3) {
  wilcox.test(values ~ parameter1)
  independence_test(values ~ parameter1,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(lor[[name]] ~ lor[[parname]]))
  print(kruskalmc(lor[[name]] ~ lor[[parname]]))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(lor[[parname1]]) - 1)) {
    for (j in (i + 1):nlevels(lor[[parname1]])) {
      cat("\nГруппы — ", levels(parameter1)[i], ", ", levels(parameter1)[j])
      ss <- subset(lor, parameter1 == levels(lor[[parname1]])[i] | parameter1 == levels(lor[[parname1]])[j])
      print(wilcox.test(ss[[name]] ~ ss[[parname1]], exact = TRUE))
      print(independence_test(ss[[name]] ~ ss[[parname1]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

## длительность заболевания

### Все

In [ ]:
name <- "длительность заболевания"
values <- lor[[name]]

In [ ]:
histogram(~values, ylab = "% от общего количества", col = "light blue", xlab = name)

In [ ]:
numSummary(values, statistics = c("mean", "sd", "IQR", "quantiles"), quantiles = c(0, .25, .5, .75, 1))
favstats(values)

In [ ]:
qqmath(~values, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = name)

In [ ]:
ad.test(values)

### пол

#### Исходные

##### Общее

In [ ]:
parname <- "пол"
parameter <- lor[[parname]]

In [ ]:
histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = name)

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = name)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, 6, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

##### Тест на нормальность

In [ ]:
for (i in 1:nlevels(lor[[parname]])) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[parname]] == levels(parameter)[i])
  print(shapiro.test(ss[[name]]))
}

In [ ]:
bartlett.test(lor[[name]] ~ parameter)

##### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))
}

##### Сравнение, распределение не нормальное

In [ ]:
wilcox.test(values ~ parameter)

In [ ]:
independence_test(values ~ parameter,
  data = lor,
  alternative = "greater",
  ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
  xtrafo = function(data) trafo(data, ordered_trafo = ff)
)

In [ ]:
independence_test(values ~ parameter,
  data = lor,
  alternative = "greater",
  ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
  xtrafo = function(data) trafo(data, ordered_trafo = ff)
)

### группа

#### Исходные

##### Общее

In [ ]:
parname <- "группа"
parameter <- lor[[parname]]

In [ ]:
histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = name)

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = name)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, 6, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

##### Тест на нормальность

In [ ]:
for (i in 1:nlevels(lor[[parname]])) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[parname]] == levels(parameter)[i])
  print(shapiro.test(ss[[name]]))
}

In [ ]:
bartlett.test(lor[[name]] ~ parameter)

##### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(lor[[parname]]) - 1)) {
    for (j in (i + 1):nlevels(lor[[parname]])) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(lor[[parname]])[i], `name`], lor[parameter == levels(lor[[parname]])[j], `name`]))
    }
  }
}

##### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(lor[[name]] ~ lor[[parname]]))
  print(kruskalmc(lor[[name]] ~ lor[[parname]]))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(lor[[parname]]) - 1)) {
    for (j in (i + 1):nlevels(lor[[parname]])) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[parname]])[i] | parameter == levels(lor[[parname]])[j])
      print(wilcox.test(ss[[name]] ~ ss[[parname]], exact = TRUE))
      print(independence_test(ss[[name]] ~ ss[[parname]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

## удаление полипов в анамнезе

### Все

In [ ]:
name <- "удаление полипов в анамнезе"
values <- lor[[name]]

In [ ]:
histogram(~values, ylab = "% от общего количества", col = "light blue", xlab = name)

In [ ]:
numSummary(values, statistics = c("mean", "sd", "IQR", "quantiles"), quantiles = c(0, .25, .5, .75, 1))
favstats(values)

In [ ]:
qqmath(~values, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = name)

In [ ]:
ad.test(values)

### пол

#### Исходные

##### Общее

In [ ]:
parname <- "пол"
parameter <- lor[[parname]]

In [ ]:
histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = name)

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = name)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")

##### Тест на нормальность

In [ ]:
for (i in 1:nlevels(lor[[parname]])) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[parname]] == levels(parameter)[i])
  print(shapiro.test(ss[[name]]))
}

In [ ]:
bartlett.test(lor[[name]] ~ parameter)

##### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))
}

##### Сравнение, распределение не нормальное

In [ ]:
wilcox.test(values ~ parameter)

In [ ]:
independence_test(values ~ parameter,
  data = lor,
  alternative = "greater",
  ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
  xtrafo = function(data) trafo(data, ordered_trafo = ff)
)

In [ ]:
independence_test(values ~ parameter,
  data = lor,
  alternative = "greater",
  ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
  xtrafo = function(data) trafo(data, ordered_trafo = ff)
)

### группа

#### Исходные

##### Общее

In [ ]:
parname <- "группа"
parameter <- lor[[parname]]

In [ ]:
histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = name)

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = name)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2.3, -0.3, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

##### Тест на нормальность

In [ ]:
for (i in 1:nlevels(lor[[parname]])) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[parname]] == levels(parameter)[i])
  print(shapiro.test(ss[[name]]))
}

In [ ]:
bartlett.test(lor[[name]] ~ parameter)

##### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(lor[[parname]]) - 1)) {
    for (j in (i + 1):nlevels(lor[[parname]])) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(lor[[parname]])[i], `name`], lor[parameter == levels(lor[[parname]])[j], `name`]))
    }
  }
}

##### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(lor[[name]] ~ lor[[parname]]))
  print(kruskalmc(lor[[name]] ~ lor[[parname]]))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(lor[[parname]]) - 1)) {
    for (j in (i + 1):nlevels(lor[[parname]])) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[parname]])[i] | parameter == levels(lor[[parname]])[j])
      print(wilcox.test(ss[[name]] ~ ss[[parname]], exact = TRUE))
      print(independence_test(ss[[name]] ~ ss[[parname]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

## группа

In [ ]:
groupping_variable <- "группа"
strata <- "пол"

### пол

In [ ]:
testing_variable <- "пол"

In [ ]:
tab.values <- xtabs(~ lor[[groupping_variable]] + lor[[testing_variable]])
dimnames(tab.values) <- setNames(dimnames(tab.values), c(groupping_variable, testing_variable))
tab.props <- round(proportions(tab.values, 1) * 100, 2)
tab.labels <- as.table(matrix(paste0(tab.values, " (", tab.props, "%)"), dim(tab.values)[1], dim(tab.values)[2]))
dimnames(tab.labels) <- dimnames(tab.values)

mosaic(tab.values,
  shade = TRUE,
  gp_labels = gpar(fontsize = 12),
  keep_aspect_ratio = FALSE,
  split_vertical = TRUE
  # , labeling=labeling_values
  , offset_varnames = c(left = 5, top = 2),
  offset_labels = c(left = 2.5, top = 1),
  rot_labels = c(left = 0, top = 0),
  margins = c(5, 2, 2, 8),
  pop = FALSE
)
labeling_cells(text = tab.labels, margin = 0)(tab.values)

#### Сравнение

##### Тест Фишера

In [ ]:
fisher.test(lor[[groupping_variable]], lor[[testing_variable]])

##### Тест Кохрейна-Мантель-Ханцеля

In [ ]:
cmh_test(~ lor[[groupping_variable]] + lor[[testing_variable]])

In [ ]:
cmh_test(~ lor[[groupping_variable]] + lor[[testing_variable]] | lor[[strata]])

## аллергия.домашняя пыль:искривление носовой перегородки

In [ ]:
for (i in match("аллергия.домашняя пыль", names(lor)):match("искривление носовой перегородки", names(lor))) {
  testing_variable <- colnames(lor)[i]

  tab.values <- xtabs(~ lor[[groupping_variable]] + lor[[testing_variable]])
  dimnames(tab.values) <- setNames(dimnames(tab.values), c(groupping_variable, testing_variable))
  tab.props <- round(proportions(tab.values, 1) * 100, 2)
  tab.labels <- as.table(matrix(paste0(tab.values, " (", tab.props, "%)"), dim(tab.values)[1], dim(tab.values)[2]))
  dimnames(tab.labels) <- dimnames(tab.values)

  display_markdown(paste0("### ", testing_variable))
  mosaic(tab.values,
    shade = TRUE,
    gp_labels = gpar(fontsize = 12),
    keep_aspect_ratio = FALSE,
    split_vertical = TRUE
    # , labeling=labeling_values
    , offset_varnames = c(left = 5, top = 2),
    offset_labels = c(left = 2.5, top = 1),
    rot_labels = c(left = 0, top = 0),
    margins = c(5, 2, 2, 8),
    pop = FALSE
  )
  labeling_cells(text = tab.labels, margin = 0)(tab.values)

  # Все
  # Тест Фишера
  cat(paste0("\n    ***Тест Фишера***\n"))
  try(print(fisher.test(lor[[groupping_variable]], lor[[testing_variable]])), silent = TRUE)

  # Тест Кохрейна-Мантель-Ханцеля
  cat(paste0("\n\n    ***Тест Кохрейна-Мантель-Ханцеля***\n"))
  try(print(cmh_test(~ lor[[groupping_variable]] + lor[[testing_variable]])), silent = TRUE)

  cat(paste0("\n\n    ***Тест Кохрейна-Мантель-Ханцеля, стратифицированный по полу***\n"))
  try(print(cmh_test(~ lor[[groupping_variable]] + lor[[testing_variable]] | lor[[strata]])), silent = TRUE)

  cat(paste0("\n\n==========\nПопарное сравнение\n"))
  # Попарно
  for (i in 1:(nlevels(lor[[groupping_variable]]) - 1)) {
    for (j in (i + 1):nlevels(lor[[groupping_variable]])) {
      cat("\nГруппы — ", levels(lor[[groupping_variable]])[i], ", ", levels(lor[[groupping_variable]])[j])
      ss <- subset(lor, lor[[groupping_variable]] == levels(lor[[groupping_variable]])[i] | lor[[groupping_variable]] == levels(lor[[groupping_variable]])[j])

      # Тест Фишера
      cat(paste0("\n    ***Тест Фишера***\n"))
      try(print(fisher.test(ss[[groupping_variable]], ss[[testing_variable]])), silent = TRUE)
    }
  }
}

## место взятия СК : аллергопроба

In [ ]:
for (i in match("место взятия СК", names(lor)):match("аллергопроба", names(lor))) {
  testing_variable <- colnames(lor)[i]

  tab.values <- xtabs(~ lor[[groupping_variable]] + lor[[testing_variable]])
  dimnames(tab.values) <- setNames(dimnames(tab.values), c(groupping_variable, testing_variable))
  tab.props <- round(proportions(tab.values, 1) * 100, 2)
  tab.labels <- as.table(matrix(paste0(tab.values, " (", tab.props, "%)"), dim(tab.values)[1], dim(tab.values)[2]))
  dimnames(tab.labels) <- dimnames(tab.values)

  display_markdown(paste0("### ", testing_variable))
  mosaic(tab.values,
    shade = TRUE,
    gp_labels = gpar(fontsize = 12),
    keep_aspect_ratio = FALSE,
    split_vertical = TRUE
    # , labeling=labeling_values
    , offset_varnames = c(left = 5, top = 2),
    offset_labels = c(left = 2.5, top = 1),
    rot_labels = c(left = 0, top = 0),
    margins = c(5, 2, 2, 8),
    pop = FALSE
  )
  labeling_cells(text = tab.labels, margin = 0)(tab.values)

  # Все
  # Тест Фишера
  cat(paste0("\n    ***Тест Фишера***\n"))
  try(print(fisher.test(lor[[groupping_variable]], lor[[testing_variable]])), silent = TRUE)

  # Тест Кохрейна-Мантель-Ханцеля
  cat(paste0("\n\n    ***Тест Кохрейна-Мантель-Ханцеля***\n"))
  try(print(cmh_test(~ lor[[groupping_variable]] + lor[[testing_variable]])), silent = TRUE)

  cat(paste0("\n\n    ***Тест Кохрейна-Мантель-Ханцеля, стратифицированный по полу***\n"))
  try(print(cmh_test(~ lor[[groupping_variable]] + lor[[testing_variable]] | lor[[strata]])), silent = TRUE)

  cat(paste0("\n\n==========\nПопарное сравнение\n"))
  # Попарно
  for (i in 1:(nlevels(lor[[groupping_variable]]) - 1)) {
    for (j in (i + 1):nlevels(lor[[groupping_variable]])) {
      cat("\nГруппы — ", levels(lor[[groupping_variable]])[i], ", ", levels(lor[[groupping_variable]])[j])
      ss <- subset(lor, lor[[groupping_variable]] == levels(lor[[groupping_variable]])[i] | lor[[groupping_variable]] == levels(lor[[groupping_variable]])[j])

      # Тест Фишера
      cat(paste0("\n    ***Тест Фишера***\n"))
      try(print(fisher.test(ss[[groupping_variable]], ss[[testing_variable]])), silent = TRUE)
    }
  }
}

## постинъекционный отек : цвет.2

In [ ]:
for (i in match("постинъекционный отек", names(lor)):match("цвет.2", names(lor))) {
  testing_variable <- colnames(lor)[i]

  tab.values <- xtabs(~ lor[[groupping_variable]] + lor[[testing_variable]])
  dimnames(tab.values) <- setNames(dimnames(tab.values), c(groupping_variable, testing_variable))
  tab.props <- round(proportions(tab.values, 1) * 100, 2)
  tab.labels <- as.table(matrix(paste0(tab.values, " (", tab.props, "%)"), dim(tab.values)[1], dim(tab.values)[2]))
  dimnames(tab.labels) <- dimnames(tab.values)

  mosaic(tab.values,
    shade = TRUE,
    gp_labels = gpar(fontsize = 12),
    keep_aspect_ratio = FALSE,
    split_vertical = TRUE
    # , labeling=labeling_values
    , offset_varnames = c(left = 5, top = 2),
    offset_labels = c(left = 2.5, top = 1),
    rot_labels = c(left = 0, top = 0),
    margins = c(5, 2, 2, 8),
    pop = FALSE
  )
  cat(paste0("\n\n============", testing_variable, "============\n"))
  labeling_cells(text = tab.labels, margin = 0)(tab.values)

  # Все
  # Тест Фишера
  cat(paste0("\n    ***Тест Фишера***\n"))
  try(print(fisher.test(lor[[groupping_variable]], lor[[testing_variable]])), silent = TRUE)

  # Тест Кохрейна-Мантель-Ханцеля
  cat(paste0("\n\n    ***Тест Кохрейна-Мантель-Ханцеля***\n"))
  try(print(cmh_test(~ lor[[groupping_variable]] + lor[[testing_variable]])), silent = TRUE)

  cat(paste0("\n\n    ***Тест Кохрейна-Мантель-Ханцеля, стратифицированный по полу***\n"))
  try(print(cmh_test(~ lor[[groupping_variable]] + lor[[testing_variable]] | lor[[strata]])), silent = TRUE)

  cat(paste0("\n\n==========\nПопарное сравнение\n"))
  # Попарно
  for (i in 1:(nlevels(lor[[groupping_variable]]) - 1)) {
    for (j in (i + 1):nlevels(lor[[groupping_variable]])) {
      cat("\nГруппы — ", levels(lor[[groupping_variable]])[i], ", ", levels(lor[[groupping_variable]])[j])
      ss <- subset(lor, lor[[groupping_variable]] == levels(lor[[groupping_variable]])[i] | lor[[groupping_variable]] == levels(lor[[groupping_variable]])[j])

      # Тест Фишера
      cat(paste0("\n    ***Тест Фишера***\n"))
      try(print(fisher.test(ss[[groupping_variable]], ss[[testing_variable]])), silent = TRUE)
    }
  }
}

## оперированные пазухи : операция на перегородке носа

In [ ]:
for (i in match("оперированные пазухи", names(lor)):match("операция на перегородке носа", names(lor))) {
  testing_variable <- colnames(lor)[i]

  tab.values <- xtabs(~ lor[[groupping_variable]] + lor[[testing_variable]])
  dimnames(tab.values) <- setNames(dimnames(tab.values), c(groupping_variable, testing_variable))
  tab.props <- round(proportions(tab.values, 1) * 100, 2)
  tab.labels <- as.table(matrix(paste0(tab.values, " (", tab.props, "%)"), dim(tab.values)[1], dim(tab.values)[2]))
  dimnames(tab.labels) <- dimnames(tab.values)

  favstats(lor[[groupping_variable]] ~ lor[[testing_variable]])

  mosaic(tab.values,
    shade = TRUE,
    gp_labels = gpar(fontsize = 12),
    keep_aspect_ratio = FALSE,
    split_vertical = TRUE
    # , labeling=labeling_values
    , offset_varnames = c(left = 5, top = 2),
    offset_labels = c(left = 2.5, top = 1),
    rot_labels = c(left = 0, top = 0),
    margins = c(5, 2, 2, 8),
    pop = FALSE
  )
  cat(paste0("\n\n============", testing_variable, "============\n"))
  labeling_cells(text = tab.labels, margin = 0)(tab.values)

  # Все
  # Тест Фишера
  cat(paste0("\n    ***Тест Фишера***\n"))
  try(print(fisher.test(lor[[groupping_variable]], lor[[testing_variable]])), silent = TRUE)

  # Тест Кохрейна-Мантель-Ханцеля
  cat(paste0("\n\n    ***Тест Кохрейна-Мантель-Ханцеля***\n"))
  try(print(cmh_test(~ lor[[groupping_variable]] + lor[[testing_variable]])), silent = TRUE)

  cat(paste0("\n\n    ***Тест Кохрейна-Мантель-Ханцеля, стратифицированный по полу***\n"))
  try(print(cmh_test(~ lor[[groupping_variable]] + lor[[testing_variable]] | lor[[strata]])), silent = TRUE)

  cat(paste0("\n\n==========\nПопарное сравнение\n"))
  # Попарно
  for (i in 1:(nlevels(lor[[groupping_variable]]) - 1)) {
    for (j in (i + 1):nlevels(lor[[groupping_variable]])) {
      cat("\nГруппы — ", levels(lor[[groupping_variable]])[i], ", ", levels(lor[[groupping_variable]])[j])
      ss <- subset(lor, lor[[groupping_variable]] == levels(lor[[groupping_variable]])[i] | lor[[groupping_variable]] == levels(lor[[groupping_variable]])[j])

      # Тест Фишера
      cat(paste0("\n    ***Тест Фишера***\n"))
      try(print(fisher.test(ss[[groupping_variable]], ss[[testing_variable]])), silent = TRUE)
    }
  }
}

## затруднение носового дыхания.0 : аносмия.3

In [ ]:
group_colors <- c("#44bec7", "#0084ff", "#ffc300", "#FAA93C", "#F99FFB", "#77EDB8", "#82003e", "#5e002d", "#000000")

for (i in match("затруднение носового дыхания.0", names(lor)):match("аносмия.3", names(lor))) {
  testing_variable <- colnames(lor)[i]

  tab.values <- xtabs(~ lor[[groupping_variable]] + lor[[testing_variable]])
  dimnames(tab.values) <- setNames(dimnames(tab.values), c(groupping_variable, testing_variable))
  tab.props <- round(proportions(tab.values, 1) * 100, 2)
  tab.labels <- as.table(matrix(paste0(tab.values, " (", tab.props, "%)"), dim(tab.values)[1], dim(tab.values)[2]))
  dimnames(tab.labels) <- dimnames(tab.values)

  # favstats(lor[[groupping_variable]] ~ lor[[testing_variable]])

  mosaic(tab.values,
    shade = TRUE,
    gp_labels = gpar(fontsize = 12),
    keep_aspect_ratio = FALSE,
    split_vertical = TRUE
    # , labeling=labeling_values
    , offset_varnames = c(left = 5, top = 2),
    offset_labels = c(left = 2.5, top = 1),
    rot_labels = c(left = 0, top = 0),
    margins = c(5, 2, 2, 8),
    pop = FALSE
  )
  cat(paste0("\n\n============", testing_variable, "============\n"))
  labeling_cells(text = tab.labels, margin = 0)(tab.values)

  # Тест Фишера
  cat(paste0("\n    ***Тест Фишера***\n"))
  try(print(fisher.test(lor[[groupping_variable]], lor[[testing_variable]])), silent = TRUE)

  # Тест Кохрейна-Мантель-Ханцеля
  cat(paste0("\n\n    ***Тест Кохрейна-Мантель-Ханцеля***\n"))
  try(print(cmh_test(~ lor[[groupping_variable]] + lor[[testing_variable]])), silent = TRUE)

  cat(paste0("\n\n    ***Тест Кохрейна-Мантель-Ханцеля, стратифицированный по полу***\n"))
  try(cmh_test(~ lor[[groupping_variable]] + lor[[testing_variable]] | lor[[strata]]), silent = TRUE)

  # Линейно-линейный ассоциативный тест
  cat(paste0("\n\n    ***ЛЛАТ***\n"))
  try(lbl_test(~ lor[[groupping_variable]] + lor[[testing_variable]]), silent = TRUE)

  cat(paste0("\n\n==========\nПопарное сравнение\n"))
  # Попарно
  for (i in 1:(nlevels(lor[[groupping_variable]]) - 1)) {
    for (j in (i + 1):nlevels(lor[[groupping_variable]])) {
      cat("\nГруппы — ", levels(lor[[groupping_variable]])[i], ", ", levels(lor[[groupping_variable]])[j])
      ss <- subset(lor, lor[[groupping_variable]] == levels(lor[[groupping_variable]])[i] | lor[[groupping_variable]] == levels(lor[[groupping_variable]])[j])

      # Тест Фишера
      cat(paste0("\n    ***Тест Фишера***\n"))
      try(print(fisher.test(ss[[groupping_variable]], ss[[testing_variable]])), silent = TRUE)
    }
  }
}

## прозраное отделяемое.0 : проходимость соустий ВЧП.3

In [ ]:
group_colors <- c("#44bec7", "#0084ff", "#ffc300", "#FAA93C", "#F99FFB", "#77EDB8", "#82003e", "#5e002d", "#000000")

for (i in match("прозраное отделяемое.0", names(lor)):match("проходимость соустий ВЧП.3", names(lor))) {
  testing_variable <- colnames(lor)[i]

  tab.values <- xtabs(~ lor[[groupping_variable]] + lor[[testing_variable]])
  dimnames(tab.values) <- setNames(dimnames(tab.values), c(groupping_variable, testing_variable))
  tab.props <- round(proportions(tab.values, 1) * 100, 2)
  tab.labels <- as.table(matrix(paste0(tab.values, " (", tab.props, "%)"), dim(tab.values)[1], dim(tab.values)[2]))
  dimnames(tab.labels) <- dimnames(tab.values)

  # favstats(lor[[groupping_variable]] ~ lor[[testing_variable]])

  mosaic(tab.values,
    shade = TRUE,
    gp_labels = gpar(fontsize = 12),
    keep_aspect_ratio = FALSE,
    split_vertical = TRUE
    # , labeling=labeling_values
    , offset_varnames = c(left = 5, top = 2),
    offset_labels = c(left = 2.5, top = 1),
    rot_labels = c(left = 0, top = 0),
    margins = c(5, 2, 2, 8),
    pop = FALSE
  )
  cat(paste0("\n\n============", testing_variable, "============\n"))
  labeling_cells(text = tab.labels, margin = 0)(tab.values)

  # Тест Фишера
  cat(paste0("\n    ***Тест Фишера***\n"))
  try(print(fisher.test(lor[[groupping_variable]], lor[[testing_variable]])), silent = TRUE)

  # Тест Кохрейна-Мантель-Ханцеля
  cat(paste0("\n\n    ***Тест Кохрейна-Мантель-Ханцеля***\n"))
  try(print(cmh_test(~ lor[[groupping_variable]] + lor[[testing_variable]])), silent = TRUE)

  cat(paste0("\n\n    ***Тест Кохрейна-Мантель-Ханцеля, стратифицированный по полу***\n"))
  try(cmh_test(~ lor[[groupping_variable]] + lor[[testing_variable]] | lor[[strata]]), silent = TRUE)

  # Линейно-линейный ассоциативный тест
  cat(paste0("\n\n    ***ЛЛАТ***\n"))
  try(lbl_test(~ lor[[groupping_variable]] + lor[[testing_variable]]), silent = TRUE)

  cat(paste0("\n\n==========\nПопарное сравнение\n"))
  # Попарно
  for (i in 1:(nlevels(lor[[groupping_variable]]) - 1)) {
    for (j in (i + 1):nlevels(lor[[groupping_variable]])) {
      cat("\nГруппы — ", levels(lor[[groupping_variable]])[i], ", ", levels(lor[[groupping_variable]])[j])
      ss <- subset(lor, lor[[groupping_variable]] == levels(lor[[groupping_variable]])[i] | lor[[groupping_variable]] == levels(lor[[groupping_variable]])[j])

      # Тест Фишера
      cat(paste0("\n    ***Тест Фишера***\n"))
      try(print(fisher.test(ss[[groupping_variable]], ss[[testing_variable]])), silent = TRUE)
    }
  }
}

## распространенность полипозного процесса КТ

In [ ]:
testing_variable <- "распространенность полипозного процесса КТ"

In [ ]:
tab.values <- xtabs(~ lor[[groupping_variable]] + lor[[testing_variable]])
dimnames(tab.values) <- setNames(dimnames(tab.values), c(groupping_variable, testing_variable))
tab.props <- round(proportions(tab.values, 1) * 100, 2)
tab.labels <- as.table(matrix(paste0(tab.values, " (", tab.props, "%)"), dim(tab.values)[1], dim(tab.values)[2]))
dimnames(tab.labels) <- dimnames(tab.values)

mosaic(tab.values,
  shade = TRUE,
  gp_labels = gpar(fontsize = 12),
  keep_aspect_ratio = FALSE,
  split_vertical = TRUE
  # , labeling=labeling_values
  , offset_varnames = c(left = 5, top = 2),
  offset_labels = c(left = 2.5, top = 1),
  rot_labels = c(left = 0, top = 0),
  margins = c(5, 2, 2, 8),
  pop = FALSE
)
labeling_cells(text = tab.labels, margin = 0)(tab.values)

### Сравнение

#### Тест Фишера

In [ ]:
fisher.test(lor[[groupping_variable]], lor[[testing_variable]])

#### Тест Кохрейна-Мантель-Ханцеля

In [ ]:
cmh_test(~ lor[[groupping_variable]] + lor[[testing_variable]])

In [ ]:
cmh_test(~ lor[[groupping_variable]] + lor[[testing_variable]] | lor[[strata]])

In [ ]:
cat(paste0("\n\n==========\nПопарное сравнение\n"))
# Попарно
for (i in 1:(nlevels(lor[[groupping_variable]]) - 1)) {
  for (j in (i + 1):nlevels(lor[[groupping_variable]])) {
    cat("\nГруппы — ", levels(lor[[groupping_variable]])[i], ", ", levels(lor[[groupping_variable]])[j])
    ss <- subset(lor, lor[[groupping_variable]] == levels(lor[[groupping_variable]])[i] | lor[[groupping_variable]] == levels(lor[[groupping_variable]])[j])

    # Тест Фишера
    cat(paste0("\n    ***Тест Фишера***\n"))
    try(print(fisher.test(ss[[groupping_variable]], ss[[testing_variable]])), silent = TRUE)
  }
}

## лобные пазухи : остиомеатальный комплекс

In [ ]:
group_colors <- c("#44bec7", "#0084ff", "#ffc300", "#FAA93C", "#F99FFB", "#77EDB8", "#82003e", "#5e002d", "#000000")

for (i in match("лобные пазухи", names(lor)):match("остиомеатальный комплекс", names(lor))) {
  testing_variable <- colnames(lor)[i]

  tab.values <- xtabs(~ lor[[groupping_variable]] + lor[[testing_variable]])
  dimnames(tab.values) <- setNames(dimnames(tab.values), c(groupping_variable, testing_variable))
  tab.props <- round(proportions(tab.values, 1) * 100, 2)
  tab.labels <- as.table(matrix(paste0(tab.values, " (", tab.props, "%)"), dim(tab.values)[1], dim(tab.values)[2]))
  dimnames(tab.labels) <- dimnames(tab.values)

  # favstats(lor[[groupping_variable]] ~ lor[[testing_variable]])

  mosaic(tab.values,
    shade = TRUE,
    gp_labels = gpar(fontsize = 12),
    keep_aspect_ratio = FALSE,
    split_vertical = TRUE
    # , labeling=labeling_values
    , offset_varnames = c(left = 5, top = 2),
    offset_labels = c(left = 2.5, top = 1),
    rot_labels = c(left = 0, top = 0),
    margins = c(5, 2, 2, 8),
    pop = FALSE
  )
  cat(paste0("\n\n============", testing_variable, "============\n"))
  labeling_cells(text = tab.labels, margin = 0)(tab.values)

  # Тест Фишера
  cat(paste0("\n    ***Тест Фишера***\n"))
  try(print(fisher.test(lor[[groupping_variable]], lor[[testing_variable]])), silent = TRUE)

  # Тест Кохрейна-Мантель-Ханцеля
  cat(paste0("\n\n    ***Тест Кохрейна-Мантель-Ханцеля***\n"))
  try(print(cmh_test(~ lor[[groupping_variable]] + lor[[testing_variable]])), silent = TRUE)

  cat(paste0("\n\n    ***Тест Кохрейна-Мантель-Ханцеля, стратифицированный по полу***\n"))
  try(cmh_test(~ lor[[groupping_variable]] + lor[[testing_variable]] | lor[[strata]]), silent = TRUE)

  # Линейно-линейный ассоциативный тест
  cat(paste0("\n\n    ***ЛЛАТ***\n"))
  try(lbl_test(~ lor[[groupping_variable]] + lor[[testing_variable]]), silent = TRUE)

  cat(paste0("\n\n==========\nПопарное сравнение\n"))
  # Попарно
  for (i in 1:(nlevels(lor[[groupping_variable]]) - 1)) {
    for (j in (i + 1):nlevels(lor[[groupping_variable]])) {
      cat("\nГруппы — ", levels(lor[[groupping_variable]])[i], ", ", levels(lor[[groupping_variable]])[j])
      ss <- subset(lor, lor[[groupping_variable]] == levels(lor[[groupping_variable]])[i] | lor[[groupping_variable]] == levels(lor[[groupping_variable]])[j])

      # Тест Фишера
      cat(paste0("\n    ***Тест Фишера***\n"))
      try(print(fisher.test(ss[[groupping_variable]], ss[[testing_variable]])), silent = TRUE)
    }
  }
}

## шкала Лунд-Маккей

### Общее

In [ ]:
parname <- "шкала Лунд-Маккей"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname, breaks = seq(0, 24, 4), xlim = c(0, 24), xlat = c(0, 24, 4))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = name)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  print(shapiro.test(ss[[parname]]))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

## отделяемое : отек

In [ ]:
group_colors <- c("#44bec7", "#0084ff", "#ffc300", "#FAA93C", "#F99FFB", "#77EDB8", "#82003e", "#5e002d", "#000000")

for (i in match("отделяемое", names(lor)):match("отек", names(lor))) {
  testing_variable <- colnames(lor)[i]

  tab.values <- xtabs(~ lor[[groupping_variable]] + lor[[testing_variable]])
  dimnames(tab.values) <- setNames(dimnames(tab.values), c(groupping_variable, testing_variable))
  tab.props <- round(proportions(tab.values, 1) * 100, 2)
  tab.labels <- as.table(matrix(paste0(tab.values, " (", tab.props, "%)"), dim(tab.values)[1], dim(tab.values)[2]))
  dimnames(tab.labels) <- dimnames(tab.values)

  # favstats(lor[[groupping_variable]] ~ lor[[testing_variable]])

  mosaic(tab.values,
    shade = TRUE,
    gp_labels = gpar(fontsize = 12),
    keep_aspect_ratio = FALSE,
    split_vertical = TRUE
    # , labeling=labeling_values
    , offset_varnames = c(left = 5, top = 2),
    offset_labels = c(left = 2.5, top = 1),
    rot_labels = c(left = 0, top = 0),
    margins = c(5, 2, 2, 8),
    pop = FALSE
  )
  cat(paste0("\n\n============", testing_variable, "============\n"))
  labeling_cells(text = tab.labels, margin = 0)(tab.values)

  # Тест Фишера
  cat(paste0("\n    ***Тест Фишера***\n"))
  try(print(fisher.test(lor[[groupping_variable]], lor[[testing_variable]])), silent = TRUE)

  # Тест Кохрейна-Мантель-Ханцеля
  cat(paste0("\n\n    ***Тест Кохрейна-Мантель-Ханцеля***\n"))
  try(print(cmh_test(~ lor[[groupping_variable]] + lor[[testing_variable]])), silent = TRUE)

  cat(paste0("\n\n    ***Тест Кохрейна-Мантель-Ханцеля, стратифицированный по полу***\n"))
  try(cmh_test(~ lor[[groupping_variable]] + lor[[testing_variable]] | lor[[strata]]), silent = TRUE)

  # Линейно-линейный ассоциативный тест
  cat(paste0("\n\n    ***ЛЛАТ***\n"))
  try(lbl_test(~ lor[[groupping_variable]] + lor[[testing_variable]]), silent = TRUE)

  cat(paste0("\n\n==========\nПопарное сравнение\n"))
  # Попарно
  for (i in 1:(nlevels(lor[[groupping_variable]]) - 1)) {
    for (j in (i + 1):nlevels(lor[[groupping_variable]])) {
      cat("\nГруппы — ", levels(lor[[groupping_variable]])[i], ", ", levels(lor[[groupping_variable]])[j])
      ss <- subset(lor, lor[[groupping_variable]] == levels(lor[[groupping_variable]])[i] | lor[[groupping_variable]] == levels(lor[[groupping_variable]])[j])

      # Тест Фишера
      cat(paste0("\n    ***Тест Фишера***\n"))
      try(print(fisher.test(ss[[groupping_variable]], ss[[testing_variable]])), silent = TRUE)
    }
  }
}

## бальная оценка

### Тест на нормальность

In [ ]:
parname <- "бальная оценка"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname, breaks = seq(0, 10, 1), xlim = c(0, 5))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = name)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  try(print(shapiro.test(ss[[parname]])))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      try(print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`])))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      try(print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE)))
      try(print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      )))
    }
  }
}

## полипы.0 : шкала итог.3

In [ ]:
group_colors <- c("#44bec7", "#0084ff", "#ffc300", "#FAA93C", "#F99FFB", "#77EDB8", "#82003e", "#5e002d", "#000000")

for (i in match("полипы.0", names(lor)):match("шкала итог.3", names(lor))) {
  testing_variable <- colnames(lor)[i]

  tab.values <- xtabs(~ lor[[groupping_variable]] + lor[[testing_variable]])
  dimnames(tab.values) <- setNames(dimnames(tab.values), c(groupping_variable, testing_variable))
  tab.props <- round(proportions(tab.values, 1) * 100, 2)
  tab.labels <- as.table(matrix(paste0(tab.values, " (", tab.props, "%)"), dim(tab.values)[1], dim(tab.values)[2]))
  dimnames(tab.labels) <- dimnames(tab.values)

  favstats(lor[[groupping_variable]] ~ lor[[testing_variable]])

  mosaic(tab.values,
    shade = TRUE,
    gp_labels = gpar(fontsize = 12),
    keep_aspect_ratio = FALSE,
    split_vertical = TRUE
    # , labeling=labeling_values
    , offset_varnames = c(left = 5, top = 2),
    offset_labels = c(left = 2.5, top = 1),
    rot_labels = c(left = 0, top = 0),
    margins = c(5, 2, 2, 8),
    pop = FALSE
  )
  cat(paste0("\n\n============", testing_variable, "============\n"))
  labeling_cells(text = tab.labels, margin = 0)(tab.values)

  # Тест Фишера
  cat(paste0("\n    ***Тест Фишера***\n"))
  try(print(fisher.test(lor[[groupping_variable]], lor[[testing_variable]])), silent = TRUE)

  # Тест Кохрейна-Мантель-Ханцеля
  cat(paste0("\n\n    ***Тест Кохрейна-Мантель-Ханцеля***\n"))
  try(print(cmh_test(~ lor[[groupping_variable]] + lor[[testing_variable]])), silent = TRUE)

  cat(paste0("\n\n    ***Тест Кохрейна-Мантель-Ханцеля, стратифицированный по полу***\n"))
  try(cmh_test(~ lor[[groupping_variable]] + lor[[testing_variable]] | lor[[strata]]), silent = TRUE)

  # Линейно-линейный ассоциативный тест
  cat(paste0("\n\n    ***ЛЛАТ***\n"))
  try(lbl_test(~ lor[[groupping_variable]] + lor[[testing_variable]]), silent = TRUE)

  cat(paste0("\n\n==========\nПопарное сравнение\n"))
  # Попарно
  for (i in 1:(nlevels(lor[[groupping_variable]]) - 1)) {
    for (j in (i + 1):nlevels(lor[[groupping_variable]])) {
      cat("\nГруппы — ", levels(lor[[groupping_variable]])[i], ", ", levels(lor[[groupping_variable]])[j])
      ss <- subset(lor, lor[[groupping_variable]] == levels(lor[[groupping_variable]])[i] | lor[[groupping_variable]] == levels(lor[[groupping_variable]])[j])

      # Тест Фишера
      cat(paste0("\n    ***Тест Фишера***\n"))
      try(print(fisher.test(ss[[groupping_variable]], ss[[testing_variable]])), silent = TRUE)
    }
  }
}

### полипы : шкала итог -- потоковые

In [ ]:
group_colors <- c("#44bec7", "#0084ff", "#ffc300", "#FAA93C", "#F99FFB", "#77EDB8", "#82003e", "#5e002d", "#000000")

for (i in match("полипы", names(lorl)):match("шкала итог", names(lorl))) {
  testing_variable <- colnames(lorl)[i]

  # Потоковые диаграммы
  default.warn <- getOption("warn")
  options(warn = -1)

  metrics <- c(testing_variable)
  build_sankey_strataG(lorl, "time", groupping_variable, metrics, group_colors, FALSE, FALSE)

  options(warn = default.warn)
}

## шкала итог0.0

In [ ]:
parname <- "шкала итог0.0"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname) # , breaks=seq(0, 24, 4), xlim=c(0,24), xlat=c(0, 24, 4))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = name)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  print(shapiro.test(ss[[parname]]))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

## шкала итог0.2

In [ ]:
parname <- "шкала итог0.2"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname) # , breaks=seq(0, 24, 4), xlim=c(0,24), xlat=c(0, 24, 4))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = name)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  print(shapiro.test(ss[[parname]]))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

## шкала итог0.3

In [ ]:
parname <- "шкала итог0.3"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname) # , breaks=seq(0, 24, 4), xlim=c(0,24), xlat=c(0, 24, 4))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = name)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  print(shapiro.test(ss[[parname]]))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

In [ ]:
for (varid in match("шкала итог0.0", names(lor)):match("шкала итог0.3", names(lor))) {
  testing_variable <- colnames(lor)[varid]

  # ------------------------------------------
  cat(paste0("\n\n=================================================\n"))
  cat(paste0("============ ", testing_variable, " ============\n"))
  cat(paste0("=================================================\n"))

  parname <- testing_variable
  values <- lor[[parname]]
  parameter <- lor[[groupping_variable]]

  # ------------------------------------------
  cat(paste0("\n\n============ #### Общее\n"))

  display(try(histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname)))

  print(numSummary(values,
    statistics = c("mean", "sd", "IQR", "quantiles"),
    quantiles = c(0, .25, .5, .75, 1), groups = parameter
  ))
  display(favstats(values ~ parameter))

  print(try(qqmath(~ values | parameter, panel = function(x, ...) {
    panel.qqmathline(x, ...)
    panel.qqmath(x, ...)
  }, ylab = name)))

  try(beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname))
  try(boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22"))
  # abline(h= (12), lty=2, col = "grey")
  par("xpd" = TRUE)
  legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
  par("xpd" = FALSE)

  # ------------------------------------------
  cat(paste0("\n\n============ #### Тест на нормальность\n"))
  for (i in 1:nlevels(parameter)) {
    cat("Группа —", levels(parameter)[i])
    ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
    try(print(shapiro.test(ss[[parname]])))
  }

  try(bartlett.test(values ~ parameter))

  # ------------------------------------------
  cat(paste0("\n\n============ #### Сравнение, нормальные распределения\n"))

  if (nlevels(parameter) < 3) {
    try(t.test(values ~ parameter))
  } else {
    agg1.aov <- try(aov(values ~ parameter))
    print(agg1.aov)
    cat()
    print(summary(agg1.aov))
    cat("\n==========\nTukey post hoc test\n")
    print(try(TukeyHSD(agg1.aov)))
    cat("\n==========\nHSD Test \n")
    print(try(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах")))

    cat("\n==========\nPairwise comparison \n")
    for (i in 1:(nlevels(parameter) - 1)) {
      for (j in (i + 1):nlevels(parameter)) {
        cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
        print(try(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`])))
      }
    }
  }

  # ------------------------------------------
  cat(paste0("\n\n============ #### Сравнение, нормальные не распределения\n"))

  if (nlevels(parameter) < 3) {
    try(wilcox.test(values ~ parameter))
    try(independence_test(values ~ parameter,
      data = lor,
      alternative = "two.sided",
      ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
      xtrafo = function(data) trafo(data, ordered_trafo = ff)
    ))
  } else {
    try(print(kruskal.test(values ~ parameter)))
    try(print(kruskalmc(values ~ parameter)))

    cat("\n==========\nPairwise comparison \n")
    for (i in 1:(nlevels(parameter) - 1)) {
      for (j in (i + 1):nlevels(parameter)) {
        cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
        ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
        try(print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE)))
        try(print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
          data = ss,
          alternative = "two.sided",
          distribution = "exact",
          ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
          xtrafo = function(data) trafo(data, ordered_trafo = ff)
        )))
      }
    }
  }
}

## шкала.итог0

In [ ]:
parname <- "шкала итог0"
values <- lorl[[parname]]
parameter <- lorl[[groupping_variable]]

In [ ]:
try(boxplot(values ~ parameter + time, data = lorl, varwidth = TRUE, outline = FALSE, xlab = paste0(groupping_variable, ", время"), ylab = parname, col = c("light yellow", "bisque", "pink")))
try(beeswarm(values ~ parameter + time, data = lorl, method = "swarm", add = TRUE, pch = 16, pwcol = пол))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(6, -1.5, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## грибы.0 : грибы.3

In [ ]:
group_colors <- c("#44bec7", "#0084ff", "#ffc300", "#FAA93C", "#F99FFB", "#77EDB8", "#82003e", "#5e002d", "#000000")

for (i in match("грибы.0", names(lor)):match("грибы.3", names(lor))) {
  testing_variable <- colnames(lor)[i]

  tab.values <- xtabs(~ lor[[groupping_variable]] + lor[[testing_variable]])
  dimnames(tab.values) <- setNames(dimnames(tab.values), c(groupping_variable, testing_variable))
  tab.props <- round(proportions(tab.values, 1) * 100, 2)
  tab.labels <- as.table(matrix(paste0(tab.values, " (", tab.props, "%)"), dim(tab.values)[1], dim(tab.values)[2]))
  dimnames(tab.labels) <- dimnames(tab.values)

  favstats(lor[[groupping_variable]] ~ lor[[testing_variable]])

  mosaic(tab.values,
    shade = TRUE,
    gp_labels = gpar(fontsize = 12),
    keep_aspect_ratio = FALSE,
    split_vertical = TRUE
    # , labeling=labeling_values
    , offset_varnames = c(left = 5, top = 2),
    offset_labels = c(left = 2.5, top = 1),
    rot_labels = c(left = 0, top = 0),
    margins = c(5, 2, 2, 8),
    pop = FALSE
  )
  cat(paste0("\n\n============", testing_variable, "============\n"))
  labeling_cells(text = tab.labels, margin = 0)(tab.values)

  # Тест Фишера
  cat(paste0("\n    ***Тест Фишера***\n"))
  try(print(fisher.test(lor[[groupping_variable]], lor[[testing_variable]])), silent = TRUE)

  # Тест Кохрейна-Мантель-Ханцеля
  cat(paste0("\n\n    ***Тест Кохрейна-Мантель-Ханцеля***\n"))
  try(print(cmh_test(~ lor[[groupping_variable]] + lor[[testing_variable]])), silent = TRUE)

  cat(paste0("\n\n    ***Тест Кохрейна-Мантель-Ханцеля, стратифицированный по полу***\n"))
  try(cmh_test(~ lor[[groupping_variable]] + lor[[testing_variable]] | lor[[strata]]), silent = TRUE)

  # Линейно-линейный ассоциативный тест
  cat(paste0("\n\n    ***ЛЛАТ***\n"))
  try(lbl_test(~ lor[[groupping_variable]] + lor[[testing_variable]]), silent = TRUE)

  cat(paste0("\n\n==========\nПопарное сравнение\n"))
  # Попарно
  for (i in 1:(nlevels(lor[[groupping_variable]]) - 1)) {
    for (j in (i + 1):nlevels(lor[[groupping_variable]])) {
      cat("\nГруппы — ", levels(lor[[groupping_variable]])[i], ", ", levels(lor[[groupping_variable]])[j])
      ss <- subset(lor, lor[[groupping_variable]] == levels(lor[[groupping_variable]])[i] | lor[[groupping_variable]] == levels(lor[[groupping_variable]])[j])

      # Тест Фишера
      cat(paste0("\n    ***Тест Фишера***\n"))
      try(print(fisher.test(ss[[groupping_variable]], ss[[testing_variable]])), silent = TRUE)
    }
  }
}

### грибы -- потоковые

In [ ]:
group_colors <- c("#44bec7", "#0084ff", "#ffc300", "#FAA93C", "#F99FFB", "#77EDB8", "#82003e", "#5e002d", "#000000")

for (i in match("грибы", names(lorl)):match("грибы", names(lorl))) {
  testing_variable <- colnames(lorl)[i]

  # Потоковые диаграммы
  default.warn <- getOption("warn")
  options(warn = -1)

  metrics <- c(testing_variable)
  build_sankey_strataG(lorl, "time", groupping_variable, metrics, group_colors, FALSE, FALSE)

  options(warn = default.warn)
}

## флора.0 : флора.3

In [ ]:
group_colors <- c("#44bec7", "#0084ff", "#ffc300", "#FAA93C", "#F99FFB", "#77EDB8", "#82003e", "#5e002d", "#000000")

for (i in match("флора.0", names(lor)):match("флора.3", names(lor))) {
  testing_variable <- colnames(lor)[i]

  tab.values <- xtabs(~ lor[[groupping_variable]] + lor[[testing_variable]])
  dimnames(tab.values) <- setNames(dimnames(tab.values), c(groupping_variable, testing_variable))
  tab.props <- round(proportions(tab.values, 1) * 100, 2)
  tab.labels <- as.table(matrix(paste0(tab.values, " (", tab.props, "%)"), dim(tab.values)[1], dim(tab.values)[2]))
  dimnames(tab.labels) <- dimnames(tab.values)

  favstats(lor[[groupping_variable]] ~ lor[[testing_variable]])

  mosaic(tab.values,
    shade = TRUE,
    gp_labels = gpar(fontsize = 12),
    keep_aspect_ratio = FALSE,
    split_vertical = TRUE
    # , labeling=labeling_values
    , offset_varnames = c(left = 5, top = 2),
    offset_labels = c(left = 2.5, top = 1),
    rot_labels = c(left = 0, top = 0),
    margins = c(5, 2, 2, 8),
    pop = FALSE
  )
  cat(paste0("\n\n============", testing_variable, "============\n"))
  labeling_cells(text = tab.labels, margin = 0)(tab.values)

  # Тест Фишера
  cat(paste0("\n    ***Тест Фишера***\n"))
  try(print(fisher.test(lor[[groupping_variable]], lor[[testing_variable]])), silent = TRUE)

  # Тест Кохрейна-Мантель-Ханцеля
  cat(paste0("\n\n    ***Тест Кохрейна-Мантель-Ханцеля***\n"))
  try(print(cmh_test(~ lor[[groupping_variable]] + lor[[testing_variable]])), silent = TRUE)

  cat(paste0("\n\n    ***Тест Кохрейна-Мантель-Ханцеля, стратифицированный по полу***\n"))
  try(cmh_test(~ lor[[groupping_variable]] + lor[[testing_variable]] | lor[[strata]]), silent = TRUE)

  # Линейно-линейный ассоциативный тест
  cat(paste0("\n\n    ***ЛЛАТ***\n"))
  try(lbl_test(~ lor[[groupping_variable]] + lor[[testing_variable]]), silent = TRUE)

  cat(paste0("\n\n==========\nПопарное сравнение\n"))
  # Попарно
  for (i in 1:(nlevels(lor[[groupping_variable]]) - 1)) {
    for (j in (i + 1):nlevels(lor[[groupping_variable]])) {
      cat("\nГруппы — ", levels(lor[[groupping_variable]])[i], ", ", levels(lor[[groupping_variable]])[j])
      ss <- subset(lor, lor[[groupping_variable]] == levels(lor[[groupping_variable]])[i] | lor[[groupping_variable]] == levels(lor[[groupping_variable]])[j])

      # Тест Фишера
      cat(paste0("\n    ***Тест Фишера***\n"))
      try(print(fisher.test(ss[[groupping_variable]], ss[[testing_variable]])), silent = TRUE)
    }
  }
}

### флора -- потоковые

In [ ]:
group_colors <- c("#44bec7", "#0084ff", "#ffc300", "#FAA93C", "#F99FFB", "#77EDB8", "#82003e", "#5e002d", "#000000")

for (i in match("флора", names(lorl)):match("флора", names(lorl))) {
  testing_variable <- colnames(lorl)[i]

  # Потоковые диаграммы
  default.warn <- getOption("warn")
  options(warn = -1)

  metrics <- c(testing_variable)
  build_sankey_strataG(lorl, "time", groupping_variable, metrics, group_colors, FALSE, FALSE)

  options(warn = default.warn)
}

## риноцитограмма.лейкоциты.0

### Общее

In [ ]:
parname <- "риноцитограмма.лейкоциты.0"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname) # , breaks=seq(0, 24, 4), xlim=c(0,24), xlat=c(0, 24, 4))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = name)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  print(shapiro.test(ss[[parname]]))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

## риноцитограмма.лейкоциты.2

### Общее

In [ ]:
parname <- "риноцитограмма.лейкоциты.2"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname) # , breaks=seq(0, 24, 4), xlim=c(0,24), xlat=c(0, 24, 4))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = name)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  print(shapiro.test(ss[[parname]]))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

## риноцитограмма.лейкоциты.3

### Общее

In [ ]:
parname <- "риноцитограмма.лейкоциты.3"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname) # , breaks=seq(0, 24, 4), xlim=c(0,24), xlat=c(0, 24, 4))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = name)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  print(shapiro.test(ss[[parname]]))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

## риноцитограмма.лейкоциты

In [ ]:
parname <- "риноцитограмма.лейкоциты"
values <- lorl[[parname]]
parameter <- lorl[[groupping_variable]]

In [ ]:
try(boxplot(values ~ parameter + time, data = lorl, varwidth = TRUE, outline = FALSE, xlab = paste0(groupping_variable, ", время"), ylab = parname, col = c("light yellow", "bisque", "pink")))
try(beeswarm(values ~ parameter + time, data = lorl, method = "swarm", add = TRUE, pch = 16, pwcol = пол))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(6, -9, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## эпителиальные клетки.0

### Общее

In [ ]:
parname <- "эпителиальные клетки.0"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname) # , breaks=seq(0, 24, 4), xlim=c(0,24), xlat=c(0, 24, 4))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = name)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  print(shapiro.test(ss[[parname]]))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

## эпителиальные клетки.2

### Общее

In [ ]:
parname <- "эпителиальные клетки.2"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname) # , breaks=seq(0, 24, 4), xlim=c(0,24), xlat=c(0, 24, 4))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = name)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  print(shapiro.test(ss[[parname]]))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

## эпителиальные клетки.3

### Общее

In [ ]:
parname <- "эпителиальные клетки.3"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname) # , breaks=seq(0, 24, 4), xlim=c(0,24), xlat=c(0, 24, 4))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = name)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  print(shapiro.test(ss[[parname]]))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

## эпителиальные клетки

In [ ]:
parname <- "эпителиальные клетки"
values <- lorl[[parname]]
parameter <- lorl[[groupping_variable]]

In [ ]:
try(boxplot(values ~ parameter + time, data = lorl, varwidth = TRUE, outline = FALSE, xlab = paste0(groupping_variable, ", время"), ylab = parname, col = c("light yellow", "bisque", "pink")))
try(beeswarm(values ~ parameter + time, data = lorl, method = "swarm", add = TRUE, pch = 16, pwcol = пол))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
try(legend(6, -1.5, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3))
par("xpd" = FALSE)

## риноцитограмма.нейтрофилы.0

### Общее

In [ ]:
parname <- "риноцитограмма.нейтрофилы.0"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname) # , breaks=seq(0, 24, 4), xlim=c(0,24), xlat=c(0, 24, 4))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = name)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  print(shapiro.test(ss[[parname]]))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

## риноцитограмма.нейтрофилы.2

### Общее

In [ ]:
parname <- "риноцитограмма.нейтрофилы.2"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname) # , breaks=seq(0, 24, 4), xlim=c(0,24), xlat=c(0, 24, 4))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = name)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  print(shapiro.test(ss[[parname]]))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

## риноцитограмма.нейтрофилы.3

### Общее

In [ ]:
parname <- "риноцитограмма.нейтрофилы.3"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname) # , breaks=seq(0, 24, 4), xlim=c(0,24), xlat=c(0, 24, 4))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = name)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  print(shapiro.test(ss[[parname]]))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

## риноцитограмма.нейтрофилы

In [ ]:
parname <- "риноцитограмма.нейтрофилы"
values <- lorl[[parname]]
parameter <- lorl[[groupping_variable]]

In [ ]:
try(boxplot(values ~ parameter + time, data = lorl, varwidth = TRUE, outline = FALSE, xlab = paste0(groupping_variable, ", время"), ylab = parname, col = c("light yellow", "bisque", "pink")))
try(beeswarm(values ~ parameter + time, data = lorl, method = "swarm", add = TRUE, pch = 16, pwcol = пол))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(6, -1.5, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## риноцитограмма.лимфоциты.0

### Общее

In [ ]:
parname <- "риноцитограмма.лимфоциты.0"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname) # , breaks=seq(0, 24, 4), xlim=c(0,24), xlat=c(0, 24, 4))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = name)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  print(shapiro.test(ss[[parname]]))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

## риноцитограмма.лимфоциты.2

### Общее

In [ ]:
parname <- "риноцитограмма.лимфоциты.2"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname) # , breaks=seq(0, 24, 4), xlim=c(0,24), xlat=c(0, 24, 4))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = name)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  print(shapiro.test(ss[[parname]]))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

## риноцитограмма.лимфоциты.3

### Общее

In [ ]:
parname <- "риноцитограмма.лимфоциты.3"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname) # , breaks=seq(0, 24, 4), xlim=c(0,24), xlat=c(0, 24, 4))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = name)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  print(shapiro.test(ss[[parname]]))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

## риноцитограмма.лимфоциты

In [ ]:
parname <- "риноцитограмма.лимфоциты"
values <- lorl[[parname]]
parameter <- lorl[[groupping_variable]]

In [ ]:
try(boxplot(values ~ parameter + time, data = lorl, varwidth = TRUE, outline = FALSE, xlab = paste0(groupping_variable, ", время"), ylab = parname, col = c("light yellow", "bisque", "pink")))
try(beeswarm(values ~ parameter + time, data = lorl, method = "swarm", add = TRUE, pch = 16, pwcol = пол))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(6, -11, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

## риноцитограмма.эозинофилы.0

### Общее

In [ ]:
parname <- "риноцитограмма.эозинофилы.0"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname) # , breaks=seq(0, 24, 4), xlim=c(0,24), xlat=c(0, 24, 4))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = name)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  print(shapiro.test(ss[[parname]]))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

## риноцитограмма.эозинофилы.2

### Общее

In [ ]:
parname <- "риноцитограмма.эозинофилы.2"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname) # , breaks=seq(0, 24, 4), xlim=c(0,24), xlat=c(0, 24, 4))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = name)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  print(shapiro.test(ss[[parname]]))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

## риноцитограмма.эозинофилы.3

### Общее

In [ ]:
parname <- "риноцитограмма.эозинофилы.3"
values <- lor[[parname]]
parameter <- lor[[groupping_variable]]

In [ ]:
histogram(~ values | parameter, ylab = "% от общего количества", col = "light blue", xlab = parname) # , breaks=seq(0, 24, 4), xlim=c(0,24), xlat=c(0, 24, 4))

In [ ]:
numSummary(values,
  statistics = c("mean", "sd", "IQR", "quantiles"),
  quantiles = c(0, .25, .5, .75, 1), groups = parameter
)
favstats(values ~ parameter)

In [ ]:
qqmath(~ values | parameter, panel = function(x, ...) {
  panel.qqmathline(x, ...)
  panel.qqmath(x, ...)
}, ylab = name)

In [ ]:
beeswarm(values ~ parameter, data = lor, method = "swarm", pch = 16, pwcol = пол, xlab = parname)
boxplot(values ~ parameter, data = lor, add = TRUE, varwidth = TRUE, outline = FALSE, col = "#0000ff22")
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(2, -1, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)

### Тест на нормальность

In [ ]:
for (i in 1:nlevels(parameter)) {
  cat("Группа —", levels(parameter)[i])
  ss <- subset(lor, lor[[groupping_variable]] == levels(parameter)[i])
  print(shapiro.test(ss[[parname]]))
}

In [ ]:
bartlett.test(values ~ parameter)

### Сравнение, нормальные распределения

In [ ]:
if (nlevels(parameter) < 3) {
  t.test(values ~ parameter)
} else {
  agg1.aov <- aov(values ~ parameter)
  print(agg1.aov)
  cat()
  print(summary(agg1.aov))
  cat("\n==========\nTukey post hoc test\n")
  print(TukeyHSD(agg1.aov))
  cat("\n==========\nHSD Test \n")
  print(HSD.test(agg1.aov, parname, group = FALSE, main = "Различия объемов\nв разных подгруппах"))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      print(t.test(lor[parameter == levels(parameter)[i], `parname`], lor[parameter == levels(parameter)[j], `parname`]))
    }
  }
}

### Сравнение, распределение не нормальное

In [ ]:
if (nlevels(parameter) < 3) {
  wilcox.test(values ~ parameter)
  independence_test(values ~ parameter,
    data = lor,
    alternative = "two.sided",
    ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
    xtrafo = function(data) trafo(data, ordered_trafo = ff)
  )
} else {
  print(kruskal.test(values ~ parameter))
  print(kruskalmc(values ~ parameter))

  cat("\n==========\nPairwise comparison \n")
  for (i in 1:(nlevels(parameter) - 1)) {
    for (j in (i + 1):nlevels(parameter)) {
      cat("\nГруппы — ", levels(parameter)[i], ", ", levels(parameter)[j])
      ss <- subset(lor, parameter == levels(lor[[groupping_variable]])[i] | parameter == levels(lor[[groupping_variable]])[j])
      print(wilcox.test(ss[[parname]] ~ ss[[groupping_variable]], exact = TRUE))
      print(independence_test(ss[[parname]] ~ ss[[groupping_variable]],
        data = ss,
        alternative = "two.sided",
        distribution = "exact",
        ytrafo = function(data) trafo(data, numeric_trafo = rank_trafo),
        xtrafo = function(data) trafo(data, ordered_trafo = ff)
      ))
    }
  }
}

## риноцитограмма.эозинофилы

In [ ]:
parname <- "риноцитограмма.эозинофилы"
values <- lorl[[parname]]
parameter <- lorl[[groupping_variable]]

In [ ]:
try(boxplot(values ~ parameter + time, data = lorl, varwidth = TRUE, outline = FALSE, xlab = paste0(groupping_variable, ", время"), ylab = parname, col = c("light yellow", "bisque", "pink")))
try(beeswarm(values ~ parameter + time, data = lorl, method = "swarm", add = TRUE, pch = 16, pwcol = пол))
# abline(h= (12), lty=2, col = "grey")
par("xpd" = TRUE)
legend(6, -4, levels(пол), col = 1:nlevels(пол), pch = 16, lty = 1, bty = "n", horiz = FALSE, xjust = 0.5, ncol = 3)
par("xpd" = FALSE)